In [ ]:
from __future__ import annotations   # has to be the very first statement

# Plate-solving a night sky: Constellation Detection (CS-GY 6643, Project 1, Q5)


**Task.** Each scene is a 3000 × 3000 grey sky plus a folder of 32 × 32 query tiles. A tile is a star of the depicted figure, another star of the same sky, or a crop from somewhere else. A tile that is present was rotated, rescaled, blurred, shifted by a fraction of a pixel and noised, but its centre still marks the answer. For each tile we output `-1` or `(x, y, m)`, and for each scene one of the 48 constellation names.

**Score per scene:** `0.25·Presence + 0.20·Localisation + 0.25·GeometricRecovery + 0.30·Identification`, averaged over scenes.

**Pipeline at a glance**

| stage | what it does | term it serves |
|---|---|---|
| star catalogue | median background, Gaussian matched filter, peak picking, plateau centroids for saturated stars | all |
| tile localisation | whole-sky rotation/zoom template sweep, then a halo-free correlation for verification; bright-centre tiles are tested only against the brightest stars | localisation |
| presence rule | high correlation, a clear lead over the runner-up, or a large z-score against random poses | presence |
| naming | every figure is fitted to the brightest stars by exhaustive two-point hypotheses, scored by a likelihood ratio and calibrated against scrambled copies of itself | identification |
| placement | undecided bright tiles are put on the named figure's node stars | geometric recovery |

In [ ]:
# ---- where is the data?  Colab (Drive) or a local participant/ folder
import glob
import os
import sys


def locate_data():
    here = os.getcwd()
    options = [os.environ.get("CONST_DATA", ""), "participant", "../participant", os.path.join(here, "participant"),
               "/content/drive/MyDrive/constellation_data/participant", "/content/participant"]
    return next((os.path.abspath(o) for o in options if o and os.path.isdir(os.path.join(o, "patterns"))), None)


try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DATA = locate_data()
if DATA is None and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = locate_data()
if DATA is None:
    print("participant/ not found: place it next to this notebook, set CONST_DATA, or pass --data")

IN_NOTEBOOK = ("ipykernel" in sys.modules) or hasattr(sys, "ps1") or IN_COLAB   # plots and evaluation run only here
print("data:", DATA, "| colab:", IN_COLAB, "| notebook:", IN_NOTEBOOK)

## 1. The data

There are 3 labelled training scenes (41, 41 and 34 tiles) and 16 validation scenes (21 to 87 tiles). The figure drawings are RGBA line art whose size, orientation, handedness and aspect ratio are arbitrary. All three kinds of tile are degraded identically, so nothing about a tile's look reveals its class. Only matching it against the sky can.

The skies are dark and heavy-tailed. Most pixels sit near the bottom of the range, a small fraction is saturated, and the background drifts across mosaic seams and nebulosity. Two things follow. The background must be removed robustly, and the bright tail must be compressed before any correlation, or a single blown-out star dominates every window.

The rules forbid anything scene-specific. Every constant below is therefore a pixel, noise-unit or score quantity, and the training scenes are used only to check the pipeline.

## 2. Settings

All tunable numbers live in one dataclass, `Params`. `tuned(base, **changes)` returns a modified copy, which the experiments at the end use.

In [ ]:
import copy
import csv
import glob
import math
import os
import re
import sys
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, replace
from functools import lru_cache

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from scipy.spatial import cKDTree
from scipy.stats import rankdata

SKY_W = SKY_H = 3000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() or 1))
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")


@dataclass
class Params:
    """Every tunable number of the solver.  All are pixel, noise-unit or score quantities;
    none refers to a particular scene."""

    # sky star catalogue
    bg_shrink: int = 4                 # background model is built on a 1/4-size image ...
    bg_median_px: int = 31             # ... with a median of this width (about 124 full-size px)
    star_psf_sigma: float = 1.2        # Gaussian matched filter for point sources
    star_k_sigma: float = 3.6          # detection threshold, robust-noise units
    star_nms_radius: int = 3
    star_cap: int = 60000
    despeckle: bool = True             # 3x3 median before detection only
    sat_value: int = 250               # pixels at or above this are saturated
    sat_min_px: int = 3                # a plateau needs this many pixels ...
    sat_min_fill: float = 0.3          # ... and must be compact (seams and spikes are not)

    # query tiles (32 x 32 patches)
    tile_bg_median: int = 9
    tile_bg_model: str = "median"      # "median" or "surface"
    tile_edge_margin: float = 0.0
    tile_psf_sigma: float = 1.0
    tile_k_sigma: float = 3.2
    tile_remove_halo: bool = True
    tile_max_neighbours: int = 8
    tile_seed_neighbours: int = 3
    anchor_radius: float = 2.5
    tile_centre: float = 15.5

    # neighbour-geometry votes (secondary candidate source)
    zoom_lo: float = 0.7               # sky px per tile px
    zoom_hi: float = 1.45
    sky_knn: int = 16
    nb_tol_px: float = 1.5
    nb_tol_rel: float = 0.06
    nb_tol_edge: float = 2.5
    flux_spread_w: float = 0.3
    mem_budget_gb: float = 20.0

    # whole-sky rotation/zoom template sweep (primary candidate source)
    search_stride: int = 1
    search_n_angles: int = 36
    search_zooms: tuple = (0.8, 0.9, 1.0, 1.12, 1.25)
    search_inner_r: float = 0.0
    search_peaks_each: int = 6
    search_keep: int = 1500
    search_batch: int = 24
    search_backend: str = "auto"       # "auto" | "cv" | "torch"

    # verification correlation
    corr_sigma: float = 1.0
    corr_stretch: str = "sqrt"         # "asinh" | "sqrt" | "none"
    corr_outer_r: float = 22.0
    corr_remove_profile: bool = True
    corr_inner_r: float = 4.0
    corr_batch: int = 65536
    screen_keep: int = 150
    coarse_dang_deg: float = 7.5
    coarse_dzoom: float = 0.10
    fine_keep: int = 12
    fine_dang_deg: float = 1.5
    fine_dzoom: float = 0.025
    fine_dxy: float = 0.6
    null_draws: int = 200
    bgcorr_star_level: float = 3.0
    bgcorr_inner_r: float = 7.0
    bgcorr_weight: float = 0.0
    whiten: bool = True
    whiten_eps: float = 0.1
    whiten_dense: bool = False

    # present / absent rule
    pres_ncc_alone: float = 0.70
    pres_ncc_floor: float = 0.55
    pres_margin: float = 0.08
    pres_bright_ncc: float = 0.55
    pres_z_alone: float = 8.0
    pres_z_weak: float = 3.0
    pres_min_support: int = 3
    pres_zbg: float = 99.0

    # figure drawings
    fig_min_pair_len: float = 0.35
    fig_pair_cap: int = 500
    fig_pairs_per_node: int = 4

    # constellation naming
    id_n_stars: int = 350
    id_n_complete: int = 600
    id_scale_lo: float = 300.0         # sky px per normalised figure unit
    id_scale_hi: float = 1500.0
    id_shift_margin: float = 300.0
    id_verify_min: int = 60
    id_verify_max: int = 400
    id_keep_per_chunk: int = 400
    id_chunk_work: float = 2.0e7
    id_tol_rel: float = 0.02
    id_tol_lo: float = 10.0
    id_tol_hi: float = 12.0
    id_seed_mult: float = 2.5
    id_soft_mult: float = 1.5
    id_dt_cell: float = 4.0
    id_sigma_div: float = 3.0
    id_affine_min: int = 4
    id_max_aniso: float = 1.3
    id_off_margin: float = 80.0
    id_off_penalty: float = 0.0
    id_max_off: int = 1
    id_max_off_frac: float = 0.4
    id_size_w: float = 0.4
    id_missing_w: float = 0.5
    id_offimg_w: float = 0.1
    id_merge_px: float = 6.0
    null_k: int = 2
    null_sd_floor: float = 1.0
    null_seed: int = 12345
    null_shrink: float = 2.0
    null_bw: float = 1.5

    # bright-star tiles and placement on the named figure
    bright_radius: float = 6.0
    bright_pool: int = 500
    bright_refine_n: int = 60
    bright_jitter_n: int = 6
    bright_jitter_px: float = 2.0
    bright_jitter_step: float = 1.0
    bright_jitter_sat_px: float = 6.0
    bright_jitter_sat_step: float = 2.0
    core_search_r: float = 8.0
    core_level: float = 0.9
    bright_always_present: bool = True
    bright_sure_margin: float = 0.08
    dense_confident_margin: float = 0.10
    node_snap_px: float = 30.0
    node_n_angles: int = 18
    node_zooms: tuple = (0.8, 0.9, 1.0, 1.12, 1.25)
    node_min_ncc: float = 0.0
    node_min_margin: float = 0.0
    node_keep_px: float = 12.0
    place_tol_mult: float = 3.0
    blown_mean: float = 150.0
    assign_blown: bool = True
    blown_flux_frac: float = 0.5
    member_px: float = 12.0

    # run control
    prefetch: int = 3
    do_identify: bool = True
    verbose: int = 1


def tuned(base: Params | None = None, **changes) -> Params:
    """A copy of `base` (defaults if None) with some fields changed."""
    return replace(base or Params(), **changes)

### CPU implementation of the whole-sky sweep

On a GPU the template sweep is a batched `conv2d`. Without one, each rotation/zoom template is correlated with `cv2.matchTemplate` (`TM_CCORR` for the numerator), divided by the local standard deviation under the same disc mask. The two backends produce the same correlation maps.

In [ ]:
CV_THREADS = int(os.environ.get("DENSE_WORKERS", "8"))


class CvSearchSky:
    """Search image (optionally average-pooled) as a numpy array, with the per-zoom local
    standard-deviation maps cached so every tile of a scene reuses them."""

    def __init__(self, dsky, p):
        self.stride = p.search_stride
        img = dsky.img
        if self.stride > 1:
            img = F.avg_pool2d(img, self.stride)
        self.img = np.ascontiguousarray(img[0, 0].cpu().numpy().astype(np.float32))
        self._std = {}

    def window_std(self, masks, r):
        key = (masks.shape, r)
        if key not in self._std:
            padded = cv2.copyMakeBorder(self.img, r, r, r, r, cv2.BORDER_CONSTANT, value=0.0)
            squared = padded * padded
            maps = []
            for m in masks:
                m = np.ascontiguousarray(m.astype(np.float32))
                area = float(m.sum())
                s1 = cv2.matchTemplate(padded, m, cv2.TM_CCORR)
                s2 = cv2.matchTemplate(squared, m, cv2.TM_CCORR)
                var = s2 - s1 * s1 / area
                np.maximum(var, 1e-3, out=var)
                maps.append(np.sqrt(var, dtype=np.float32))
            self._std[key] = (padded, maps)
        return self._std[key]


def sweep_cv(tile_c, sky, p, dev):
    """Correlate every rotation/zoom template with the whole sky using cv2.matchTemplate
    (numerator) divided by the masked window std; keep the strongest peaks."""
    bank, masks, angles, zooms = template_bank(tile_c, p, dev)
    bank = bank[:, 0].cpu().numpy().astype(np.float32)
    masks = masks[:, 0].cpu().numpy().astype(np.float32)
    n_zoom = len(p.search_zooms)
    r = bank.shape[-1] // 2
    padded, std_maps = sky.window_std(masks, r)
    width = sky.img.shape[1]
    k = p.search_peaks_each
    nms_kernel = np.ones((5, 5), np.uint8)

    def one_template(i):
        ncc = cv2.matchTemplate(padded, np.ascontiguousarray(bank[i]), cv2.TM_CCORR) / std_maps[i % n_zoom]
        idx = np.flatnonzero(ncc >= cv2.dilate(ncc, nms_kernel))
        val = ncc.ravel()[idx]
        if len(idx) > k:
            best = np.argpartition(-val, k - 1)[:k]
            idx, val = idx[best], val[best]
        return (val.astype(np.float32), idx // width, idx % width,
                np.full(len(idx), float(angles[i]), np.float32), np.full(len(idx), float(zooms[i]), np.float32))

    with ThreadPoolExecutor(max_workers=CV_THREADS) as pool:
        parts = list(pool.map(one_template, range(len(bank))))
    val, ys, xs, ang, zm = (np.concatenate(col) for col in zip(*parts))
    n = min(p.search_keep, len(val))
    keep = np.argpartition(-val, n - 1)[:n] if len(val) > n else np.arange(len(val))
    half_px = (sky.stride - 1) / 2.0
    as_t = lambda a: torch.as_tensor(a, device=dev)
    return (as_t(xs[keep].astype(np.float32) * sky.stride + half_px), as_t(ys[keep].astype(np.float32) * sky.stride + half_px),
            as_t(zm[keep]), as_t(ang[keep]), as_t(val[keep]))

In [ ]:
def say(p, *msg):
    if p.verbose:
        print(*msg, file=sys.stderr, flush=True)


def mad_sigma(values) -> float:
    """1.4826 x median absolute deviation (subsampled for big arrays)."""
    v = np.asarray(values, np.float32).ravel()
    if v.size > 400000:
        v = v[:: int(v.size // 400000) + 1]
    centre = np.median(v)
    return float(1.4826 * np.median(np.abs(v - centre)) + 1e-6)


def nms_points(xy, radius, rank):
    """Keep the highest-ranked point of every cluster of radius `radius`."""
    n = len(xy)
    keep = np.zeros(n, bool)
    if n == 0:
        return keep
    tree = cKDTree(xy)
    dead = np.zeros(n, bool)
    for i in np.argsort(-rank):
        if dead[i]:
            continue
        keep[i] = True
        dead[tree.query_ball_point(xy[i], radius)] = True
    return keep


def procrustes(src, dst, flip):
    """Least-squares dst ~ A src + t with A = scale * rotation * diag(1, flip)."""
    mirror = np.array([1.0, float(flip)])
    s_m = src * mirror
    c_src, c_dst = s_m.mean(0), dst.mean(0)
    a, b = s_m - c_src, dst - c_dst
    U, sv, Vt = np.linalg.svd(a.T @ b)
    sgn = np.sign(np.linalg.det(Vt.T @ U.T)) or 1.0
    D = np.diag([1.0, sgn])
    R = Vt.T @ D @ U.T
    scale = (sv * np.diag(D)).sum() / ((a ** 2).sum() + 1e-12)
    A = scale * R @ np.diag(mirror)
    return A, c_dst - A @ src.mean(0)


def lstsq_affine(src, dst):
    X = np.hstack([src, np.ones((len(src), 1))])
    M, *_ = np.linalg.lstsq(X, dst, rcond=None)
    return M[:2].T, M[2]


def warp_pts(A, t, pts):
    return pts @ A.T + t


@lru_cache(maxsize=None)
def tile_offsets(c=15.5):
    """(dx, dy, radius) of every pixel of a 32 x 32 tile relative to its centre."""
    u = np.arange(32, dtype=np.float32) - c
    dx, dy = np.meshgrid(u, u)
    return dx, dy, np.hypot(dx, dy)

## 3. Star catalogue

**Background.** The sky is shrunk 4× by area averaging, median-filtered with a 31-pixel window, scaled back up and subtracted. A median suits this because stars are sparse outliers on a piecewise-flat background. A 3×3 median then removes hot pixels, for detection only.

**Detection.** Convolving the background-free sky with a Gaussian (σ = 1.2 px) is the matched filter for a point source. Peaks above 3.6 robust-noise units (1.4826 × MAD) that equal their own dilation are kept. Position and flux come from 5×5 box sums.

**Saturated stars.** A blown-out star has a flat top, so its box centroid can land anywhere on it. Such stars are moved to the centroid of their compact saturated region, and their brightness is taken as the number of saturated pixels nearby.

**Correlation image.** The background-free sky is lightly smoothed and passed through `sign(x)·sqrt(|x|)` in noise units. This keeps brightness order but stops one bright star from dominating a window.

**Tiles** get the same treatment at their own scale. A small median plus Gaussian background removes illumination drift, and the ring-average profile about the centre is subtracted so the central star's halo does not hide its neighbours. The star within 2.5 px of the centre is the anchor. Up to 8 neighbours with distance, angle, flux and an SNR weight form the tile's descriptor. The halo radius (where the ring profile drops below 25 % of the central excess) marks a tile as a *bright-star tile* when it reaches 6 px.

In [ ]:
def local_maxima(resp, signal, thr, nms_r, cap, border=2, win=5):
    """Peaks of `resp` above `thr`, with intensity-weighted centroid and flux measured on
    `signal` in a win x win box; strongest `cap` peaks returned."""
    kernel = np.ones((2 * nms_r + 1, 2 * nms_r + 1), np.uint8)
    is_peak = (resp >= cv2.dilate(resp, kernel)) & (resp > thr)
    if border > 0:
        is_peak[:border, :] = False
        is_peak[-border:, :] = False
        is_peak[:, :border] = False
        is_peak[:, -border:] = False
    ys, xs = np.nonzero(is_peak)
    if len(xs) == 0:
        return dict(x=np.zeros(0), y=np.zeros(0), flux=np.zeros(0), peak=np.zeros(0))
    weight = np.maximum(signal, 0).astype(np.float32)
    h, w = signal.shape
    gx = np.tile(np.arange(w, dtype=np.float32)[None, :], (h, 1))
    gy = np.tile(np.arange(h, dtype=np.float32)[:, None], (1, w))
    box = lambda im: cv2.boxFilter(im, -1, (win, win), normalize=False)
    m0, mx, my = box(weight), box(weight * gx), box(weight * gy)
    flux = m0[ys, xs] + 1e-6
    cx, cy = mx[ys, xs] / flux, my[ys, xs] / flux
    height = resp[ys, xs]
    runaway = (np.abs(cx - xs) > win) | (np.abs(cy - ys) > win)   # plateau: fall back to the pixel
    cx[runaway], cy[runaway] = xs[runaway], ys[runaway]
    pts = np.stack([cx, cy], 1)
    k = nms_points(pts, nms_r, height)
    pts, flux, height = pts[k], flux[k], height[k]
    order = np.argsort(-height)[:cap]
    return dict(x=pts[order, 0], y=pts[order, 1], flux=flux[order], peak=height[order])


def snap_to_plateaus(img8, hp, stars, p):
    """Move saturated stars to the centroid of their compact saturated plateau and merge
    detections that share one.  Adds x0/y0, sat_area and sat_flux."""
    n = len(stars["x"])
    stars["x0"], stars["y0"] = stars["x"].copy(), stars["y"].copy()
    stars["sat_area"] = np.zeros(n, np.float32)
    stars["sat_flux"] = np.zeros(n, np.float32)
    if n == 0 or p.sat_value <= 0:
        return stars
    blown = (img8 >= p.sat_value).astype(np.uint8)
    if not blown.any():
        return stars
    _, lab, st, ce = cv2.connectedComponentsWithStats(blown, connectivity=8)
    h, w = img8.shape
    xi = np.clip(np.round(stars["x"]).astype(int), 0, w - 1)
    yi = np.clip(np.round(stars["y"]).astype(int), 0, h - 1)
    padded = np.pad(lab, 1)
    around = np.stack([padded[yi + dy + 1, xi + dx + 1] for dy in (-1, 0, 1) for dx in (-1, 0, 1)], 1)
    region = np.where(around[:, 4] > 0, around[:, 4], around.max(1))
    area = st[region, cv2.CC_STAT_AREA]
    fill = area / np.maximum(st[region, cv2.CC_STAT_WIDTH] * st[region, cv2.CC_STAT_HEIGHT], 1)
    ok = (region > 0) & (area >= p.sat_min_px) & (fill >= p.sat_min_fill)
    if not ok.any():
        return stars
    region_flux = np.bincount(lab.ravel(), np.maximum(hp, 0).ravel().astype(np.float64), st.shape[0])
    r_ok = region[ok]
    stars["x"][ok], stars["y"][ok] = ce[r_ok, 0], ce[r_ok, 1]
    stars["sat_area"][ok] = st[r_ok, cv2.CC_STAT_AREA]
    stars["sat_flux"][ok] = region_flux[r_ok]
    keep, seen = np.ones(n, bool), set()
    for i in np.flatnonzero(ok):
        if region[i] in seen:
            keep[i] = False
        seen.add(region[i])
    return {k: v[keep] for k, v in stars.items()}


def stretch(img, mode="asinh"):
    """Divide by the robust noise and compress the bright tail."""
    x = img / mad_sigma(img if img.size < 400000 else img[::7, ::7])
    if mode == "asinh":
        return np.arcsinh(x).astype(np.float32)
    if mode == "sqrt":
        return (np.sign(x) * np.sqrt(np.abs(x))).astype(np.float32)
    return x.astype(np.float32)


def extract_sky(img8, p):
    """Star catalogue of a sky, the stretched image used for correlation, and the noise level."""
    h, w = img8.shape
    coarse = cv2.resize(img8, (w // p.bg_shrink, h // p.bg_shrink), interpolation=cv2.INTER_AREA)
    bg = cv2.resize(cv2.medianBlur(coarse, p.bg_median_px | 1), (w, h), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    hp = img8.astype(np.float32) - bg
    hp_clean = cv2.medianBlur(hp, 3) if p.despeckle else hp
    resp = cv2.GaussianBlur(hp_clean, (0, 0), p.star_psf_sigma)
    noise = mad_sigma(resp[::7, ::7])
    stars = local_maxima(resp, hp_clean, p.star_k_sigma * noise, p.star_nms_radius, p.star_cap, border=3)
    stars = snap_to_plateaus(img8, hp, stars, p)
    blown_count = cv2.boxFilter((img8 >= 250).astype(np.float32), -1, (25, 25), normalize=False)
    yi = np.clip(np.round(stars["y"]).astype(int), 0, h - 1)
    xi = np.clip(np.round(stars["x"]).astype(int), 0, w - 1)
    stars["sat"] = blown_count[yi, xi].astype(np.float32)     # brightness proxy for blown-out stars
    sky_c = stretch(cv2.GaussianBlur(hp, (0, 0), p.corr_sigma), p.corr_stretch)
    return stars, sky_c, noise


def fit_bg_surface(t):
    """Iteratively clipped quadratic surface through the non-star pixels of a tile."""
    h, w = t.shape
    yy, xx = np.mgrid[0:h, 0:w]
    X = np.stack([np.ones(h * w), xx.ravel(), yy.ravel(), xx.ravel() ** 2, yy.ravel() ** 2, (xx * yy).ravel()], 1) / 32.0
    z = t.ravel()
    use = np.ones(h * w, bool)
    for _ in range(4):
        coef, *_ = np.linalg.lstsq(X[use], z[use], rcond=None)
        model = X @ coef
        resid = z - model
        use = resid < 2.0 * mad_sigma(resid[use])
    return model.reshape(h, w).astype(np.float32)


def extract_tile_stars(t8, p):
    t = t8.astype(np.float32)
    if p.tile_bg_model == "surface":
        bg = fit_bg_surface(t)
    else:
        bg = cv2.GaussianBlur(cv2.medianBlur(t8, p.tile_bg_median | 1).astype(np.float32), (0, 0), 3)
    hp = cv2.medianBlur(t - bg, 3)
    if p.tile_remove_halo:
        _, _, rad = tile_offsets(p.tile_centre)
        ring = np.minimum(np.round(rad).astype(int), 23)
        ring_mean = np.bincount(ring.ravel(), hp.ravel(), 24) / np.maximum(np.bincount(ring.ravel(), None, 24), 1)
        hp_det = hp - ring_mean[ring]
    else:
        hp_det = hp
    resp = cv2.GaussianBlur(hp_det, (0, 0), p.tile_psf_sigma)
    noise = mad_sigma(resp)
    found = local_maxima(resp, hp_det, p.tile_k_sigma * noise, 2, 24, border=0)
    found["snr"] = found["peak"] / noise
    m, n = p.tile_edge_margin, t8.shape[0]
    inside = (found["x"] >= m) & (found["x"] <= n - 1 - m) & (found["y"] >= m) & (found["y"] <= n - 1 - m)
    found = {k: v[inside] for k, v in found.items()}
    tile_c = stretch(cv2.GaussianBlur(hp, (0, 0), p.corr_sigma), p.corr_stretch)
    return found, tile_c


@dataclass
class TileInfo:
    nb_r: np.ndarray          # neighbour distances from the anchor (brightest first)
    nb_theta: np.ndarray
    nb_flux: np.ndarray
    nb_weight: np.ndarray
    nb_edge: np.ndarray       # neighbour touches the tile border
    halo_r: float             # visible radius of the centre star
    mean: float               # mean raw intensity
    core_flux: float
    has_anchor: bool
    img_c: np.ndarray         # stretched, background-free tile
    raw: np.ndarray = None    # original 8-bit tile


def describe_tile(t8, p) -> TileInfo:
    found, tile_c = extract_tile_stars(t8, p)
    c, n = p.tile_centre, t8.shape[0]
    xy = np.stack([found["x"], found["y"]], 1) if len(found["x"]) else np.zeros((0, 2))
    anchor = -1
    if len(xy):
        d = np.hypot(xy[:, 0] - c, xy[:, 1] - c)
        if d.min() <= p.anchor_radius:
            anchor = int(np.argmin(d))
    has_anchor = anchor >= 0
    if has_anchor:
        ax, ay, core_flux = xy[anchor, 0], xy[anchor, 1], float(found["flux"][anchor])
    else:
        ax = ay = c
        core_flux = float(np.maximum(tile_c[14:18, 14:18], 0).sum()) + 1.0
    others = np.ones(len(xy), bool)
    if has_anchor:
        others[anchor] = False
    dx, dy = xy[others, 0] - ax, xy[others, 1] - ay
    r = np.hypot(dx, dy)
    far = r >= 2.0
    xs, ys = xy[others, 0][far], xy[others, 1][far]
    dx, dy, r = dx[far], dy[far], r[far]
    flux, snr = found["flux"][others][far], found["snr"][others][far]
    order = np.argsort(-flux)[: p.tile_max_neighbours]
    dx, dy, r, flux, xs, ys, snr = (a[order] for a in (dx, dy, r, flux, xs, ys, snr))
    theta = np.arctan2(dy, dx)
    weight = 0.4 + 0.6 * np.minimum(1.0, snr / 10.0)
    edge = (xs < 2) | (xs > n - 3) | (ys < 2) | (ys > n - 3)
    weight = np.where(edge, 0.6 * weight, weight)
    # halo radius: where the ring mean falls below 25% of the central excess
    _, _, rad = tile_offsets(p.tile_centre)
    profile = np.array([np.mean(tile_c[(rad >= max(rr - 0.5, 0.0)) & (rad < rr + 0.5)]) for rr in range(16)])
    floor = np.median(profile[12:16])
    peak = max(np.nanmax(profile[0:3]) - floor, 1e-3)
    halo = 3.0
    for rr in range(3, 12):
        if profile[rr] - floor < 0.25 * peak:
            break
        halo = float(rr + 1)
    return TileInfo(r.astype(np.float32), theta.astype(np.float32), flux.astype(np.float32), weight.astype(np.float32),
                    edge, min(halo, 11.0), float(t8.mean()), core_flux, has_anchor, tile_c, t8)


class StarField:
    """Catalogue positions plus, for every star, its K nearest neighbours within R."""

    def __init__(self, stars, p, R):
        self.xy = np.stack([stars["x"], stars["y"]], 1).astype(np.float32)
        self.flux = stars["flux"].astype(np.float32)
        self.N = len(self.xy)
        self.sat = stars["sat"].astype(np.float32) if "sat" in stars else np.zeros(self.N, np.float32)
        self.sat_area = np.asarray(stars.get("sat_area", np.zeros(self.N)), np.float32)
        self.xy0 = np.stack([stars["x0"], stars["y0"]], 1).astype(np.float32) if "x0" in stars else self.xy.copy()
        self.tree = cKDTree(self.xy)
        d, idx = self.tree.query(self.xy, k=p.sky_knn + 1, distance_upper_bound=R)
        d, idx = d[:, 1:], idx[:, 1:]
        valid = np.isfinite(d)
        idx = np.where(valid, idx, 0)
        dx = self.xy[idx, 0] - self.xy[:, None, 0]
        dy = self.xy[idx, 1] - self.xy[:, None, 1]
        dx[~valid] = 1e6
        dy[~valid] = 1e6
        self.nb_idx = idx
        self.nb_dx, self.nb_dy = dx.astype(np.float32), dy.astype(np.float32)
        self.nb_r = np.where(valid, np.hypot(dx, dy), np.nan).astype(np.float32)
        self.nb_phi = np.arctan2(dy, dx).astype(np.float32)
        self.nb_valid = valid
        self.nb_flux = np.where(valid, self.flux[idx], np.nan).astype(np.float32)

## 4. Figure drawings as point sets

The stars of a drawing are its white dots: pixels with alpha > 100 and all channels > 150. Connected blobs of 4 to 400 px become nodes. A blob that is much larger than typical, or elongated, is two touching dots and is split at the maxima of its distance transform. Each dot's area is kept as a size, because larger dots mark brighter stars.

Nodes are normalised to centroid 0 and RMS radius 1. Hypotheses are seeded from node pairs, at most 4 per node (the longest), since screening cost grows with pairs × nodes.

In [ ]:
@dataclass
class Figure:
    name: str
    raw: np.ndarray       # node pixel positions in the drawing
    nodes: np.ndarray     # centroid 0, RMS radius 1
    sizes: np.ndarray     # dot areas
    pairs: np.ndarray     # node pairs used to seed hypotheses
    shape: tuple


def figure_mask(im):
    if im.ndim == 3 and im.shape[2] == 4:
        alpha = im[:, :, 3]
        gray = cv2.cvtColor(im[:, :, :3], cv2.COLOR_BGR2GRAY)
        if alpha.min() < 128 < alpha.max():
            return (alpha > 128).astype(np.uint8)
    elif im.ndim == 3:
        gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    else:
        gray = im
    edge = np.concatenate([gray[0], gray[-1], gray[:, 0], gray[:, -1]])
    vals, counts = np.unique(edge, return_counts=True)
    background = int(vals[counts.argmax()])
    return (np.abs(gray.astype(int) - background) > 40).astype(np.uint8)


def voting_pairs(norm, p):
    """Long enough node pairs, at most fig_pairs_per_node x nodes (longest first)."""
    a, b = np.triu_indices(len(norm), 1)
    if len(a) == 0:
        return np.zeros((0, 2), int)
    L = np.hypot(*(norm[a] - norm[b]).T)
    ok = L >= p.fig_min_pair_len
    if ok.sum() < 3:
        ok = L >= np.percentile(L, 30)
    a, b, L = a[ok], b[ok], L[ok]
    cap = min(p.fig_pair_cap, max(12, p.fig_pairs_per_node * len(norm)))
    if len(a) > cap:
        longest = np.argsort(-L)[:cap]
        a, b = a[longest], b[longest]
    return np.stack([a, b], 1) if len(a) else np.zeros((0, 2), int)


def _split_blob(lab, i, x, y, w, h):
    """Two touching dots: split at the maxima of the blob's distance transform."""
    sub = np.pad((lab[y:y + h, x:x + w] == i).astype(np.uint8), 1)
    dt = cv2.distanceTransform(sub, cv2.DIST_L2, 5)
    ys, xs = np.nonzero((dt >= cv2.dilate(dt, np.ones((3, 3), np.uint8))) & (dt >= 0.6 * dt.max()) & (dt > 0))
    if len(xs) < 2:
        return None
    pts = np.stack([xs, ys], 1).astype(float)
    keep = nms_points(pts, max(4.0, 1.2 * float(dt.max())), dt[ys, xs])
    return pts[keep] - 1.0 + np.array([x, y], float) if keep.sum() >= 2 else None


def _generic_nodes(mask):
    """Fallback for drawings without white dots: filled discs, rings, skeleton ends."""
    from skimage.morphology import skeletonize
    H, W = mask.shape
    nodes, sizes = [], []
    dt = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    skel = skeletonize(mask.astype(bool)) if mask.any() else None
    lw = float(2 * np.median(dt[skel])) if (skel is not None and skel.any()) else 2.0
    lw = max(lw, 1.0)
    r = int(max(2, round(0.8 * lw + 1)))
    opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1)))
    n, _, st, ce = cv2.connectedComponentsWithStats(opened)
    for i in range(1, n):
        x, y, w, h, area = st[i]
        if max(w, h) <= 6 * lw + 10 and area >= 0.45 * w * h:
            nodes.append(ce[i]); sizes.append(area)
    n2, _, st2, ce2 = cv2.connectedComponentsWithStats((1 - mask).astype(np.uint8))
    for i in range(1, n2):
        x, y, w, h, area = st2[i]
        if x > 0 and y > 0 and x + w < W and y + h < H and max(w, h) <= 4 * lw + 8 and area >= 0.5 * w * h:
            nodes.append(ce2[i]); sizes.append(area + math.pi * lw * (math.sqrt(area / math.pi) + lw / 2))
    if len(nodes) < 3 and skel is not None and skel.any():
        sk = skel.astype(np.uint8)
        deg = cv2.filter2D(sk, -1, np.ones((3, 3), np.uint8), borderType=cv2.BORDER_CONSTANT) - sk
        pts = np.argwhere((sk > 0) & ((deg == 1) | (deg >= 3)))[:, ::-1].astype(float)
        if len(pts):
            for q in pts[nms_points(pts, 2 * lw + 2, np.ones(len(pts)))]:
                nodes.append(q); sizes.append(lw * lw)
    return nodes, sizes, lw


def read_figure(path, p) -> Figure:
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    mask = figure_mask(im)
    nodes, sizes = [], []
    if im.ndim == 3:
        alpha = im[:, :, 3] if im.shape[2] == 4 else np.full(mask.shape, 255, np.uint8)
        dots = ((alpha > 100) & (im[:, :, :3].astype(int).min(2) > 150)).astype(np.uint8)
        n, lab, st, ce = cv2.connectedComponentsWithStats(dots)
        blobs = [(i,) + tuple(st[i]) for i in range(1, n) if 4 <= st[i][4] <= 400 and max(st[i][2], st[i][3]) <= 30]
        typical = float(np.median([b[5] for b in blobs])) if blobs else 0.0
        for i, x, y, w, h, area in blobs:
            parts = None
            if area > 1.6 * typical or max(w, h) / max(1.0, float(min(w, h))) > 1.35:
                parts = _split_blob(lab, i, x, y, w, h)
            if parts is None:
                nodes.append(ce[i]); sizes.append(area)
            else:
                for q in parts:
                    nodes.append(q); sizes.append(area / len(parts))
    if len(nodes) >= 2:
        lw = 2.0
    else:
        nodes, sizes, lw = _generic_nodes(mask)
    nodes = np.array(nodes, float).reshape(-1, 2)
    sizes = np.array(sizes, float)
    if len(nodes):
        k = nms_points(nodes, 1.5 * lw + 1, sizes)
        nodes, sizes = nodes[k], sizes[k]
    if len(nodes) >= 2:
        c = nodes.mean(0)
        norm = (nodes - c) / (math.sqrt(((nodes - c) ** 2).sum(1).mean()) + 1e-9)
    else:
        norm = nodes.copy()
    name = os.path.basename(path).replace("_pattern.png", "").replace(".png", "")
    return Figure(name, nodes, norm, sizes, voting_pairs(norm, p), mask.shape)


def read_figures(root, p):
    figs = []
    for path in sorted(glob.glob(os.path.join(root, "patterns", "*.png"))):
        f = read_figure(path, p)
        figs.append(f)
        say(p, f"  figure {f.name:18s} nodes={len(f.nodes):3d} pairs={len(f.pairs):4d}")
    return figs

## 5. Naming the constellation

The point set is the 350 brightest stars plus the tiles judged present, merged within 6 px. It is mostly clutter.

**Hypotheses.** Two point correspondences fix a similarity transform. Every seed node pair is matched against every ordered pair of points, in both handednesses. Instead of RANSAC's random sampling, this exhaustive search is screened in bulk on the device. Each projected node contributes `exp(-½ (d / 1.5·tol)²)`, where `d` comes from a distance raster of the point set, so tight fits outrank ones that merely land near points.

**Verification.** The best 60 to 400 distinct hypotheses are matched one-to-one within 10 to 12 px and refitted. An affine fit is allowed when there are enough inliers, but only while it stays nearly isotropic and keeps its scale.

**Score.** Each inlier earns `g = log(area / (m·2πσ²))` minus its squared residual, and the free inliers of the fit are subtracted. Two further terms apply. An in-image node with none of the 600 brightest stars nearby costs up to `0.5·g`, and an off-image node costs `0.1·g`. A positive rank correlation between dot size and star brightness adds a bonus.

**Calibration.** Small figures fit random stars easily. Each figure is therefore also matched on the same points with two scrambled copies (every node rotated by its own random angle), and the reported score is `max(raw, 0) − null mean − null sd`, pooled over figures with a similar node count.

In [ ]:
@dataclass
class Hypothesis:
    name: str = "unknown"
    score: float = -1e9
    n_in: int = 0
    n_nodes: int = 0
    A: np.ndarray = None
    t: np.ndarray = None
    in_nodes: np.ndarray = None
    in_pts: np.ndarray = None
    tol: float = 0.0
    resid: float = 0.0
    raw: float = 0.0          # max(LLR, 0) before calibration
    null_mean: float = 0.0
    null_sd: float = 0.0
    n_checked: int = 0


def pair_up(A, t, fig, Q, treeQ, tol):
    """Project the nodes and match them one-to-one to Q points within tol, nearest first."""
    P = warp_pts(A, t, fig.nodes)
    d, j = treeQ.query(P)
    close = np.flatnonzero(d <= tol)
    close = close[np.argsort(d[close])]
    taken, nodes_in, pts_in = set(), [], []
    for i in close:
        if j[i] not in taken:
            taken.add(j[i])
            nodes_in.append(i)
            pts_in.append(j[i])
    nodes_in, pts_in = np.array(nodes_in, int), np.array(pts_in, int)
    return nodes_in, pts_in, (d[nodes_in] if len(nodes_in) else np.zeros(0)), P


def llr_score(fig, nodes_in, pts_in, dists, P, tol, dof, Q, qmag, p, ctx=None):
    """Log-likelihood ratio of 'figure here' vs 'random stars': each inlier earns
    g - residual^2 / 2 sigma^2 with g = log(area / (m 2 pi sigma^2)); the dof free inliers
    are removed; unmatched in-image nodes without a bright star cost up to id_missing_w g;
    off-image nodes cost id_offimg_w g; matching dot sizes to star brightness earns a bonus."""
    n, m = len(fig.nodes), len(Q)
    if m == 0:
        return -1e9
    lo, hi = Q.min(0), Q.max(0)
    area = max((hi[0] - lo[0] + 2 * tol) * (hi[1] - lo[1] + 2 * tol), 4e5)
    sigma = tol / p.id_sigma_div
    g = math.log(area / (m * 2 * math.pi * sigma * sigma))
    score = float((g - dists ** 2 / (2 * sigma * sigma)).sum()) - dof * g
    mg = p.id_off_margin
    far_out = int(((P[:, 0] < -mg) | (P[:, 0] > SKY_W + mg) | (P[:, 1] < -mg) | (P[:, 1] > SKY_H + mg)).sum())
    score -= p.id_off_penalty * g * far_out
    in_img = (P[:, 0] >= 5) & (P[:, 0] < SKY_W - 5) & (P[:, 1] >= 5) & (P[:, 1] < SKY_H - 5)
    score -= p.id_offimg_w * g * float((~in_img).sum())
    lonely = np.ones(n, bool)
    lonely[nodes_in] = False
    lonely &= in_img
    if lonely.any() and p.id_missing_w > 0:
        bright_tree = (ctx or {}).get("bright_tree")
        if bright_tree is None:
            bright_tree = cKDTree(Q)
        d, _ = bright_tree.query(P[lonely])
        score -= p.id_missing_w * g * float(np.clip((d - tol) / (2.0 * tol), 0.0, 1.0).sum())
    if (len(nodes_in) >= 4 and fig.sizes.std() > 0.15 * (fig.sizes.mean() + 1e-9)
            and qmag is not None and p.id_size_w > 0):
        c = np.corrcoef(rankdata(fig.sizes[nodes_in]), rankdata(qmag[pts_in]))[0, 1]
        if np.isfinite(c):
            score += p.id_size_w * g * c
    return float(score)


def polish(fig, A, t, flip, Q, qmag, treeQ, p, ctx=None) -> Hypothesis:
    """Refit a seed transform on its inliers (affine when enough, else similarity) while the
    fit improves, rejecting fits that drift in scale or become anisotropic."""
    s0 = math.sqrt(abs(np.linalg.det(A)))
    tol = min(p.id_tol_hi, max(p.id_tol_lo, p.id_tol_rel * s0))
    nodes_in, pts_in, dd, P = pair_up(A, t, fig, Q, treeQ, p.id_seed_mult * tol)
    dof, accepted = 2, False
    for _ in range(6):
        if len(nodes_in) < 3:
            break
        options = []
        if len(nodes_in) >= p.id_affine_min:
            options.append((3,) + lstsq_affine(fig.nodes[nodes_in], Q[pts_in]))
        options.append((2,) + procrustes(fig.nodes[nodes_in], Q[pts_in], flip))
        chosen = None
        for dof2, Ac, tc in options:
            sv = np.linalg.svd(Ac, compute_uv=False)
            s2 = math.sqrt(sv[0] * sv[1])
            if sv[1] >= 1e-9 and sv[0] / sv[1] <= p.id_max_aniso and 0.7 * s0 <= s2 <= 1.4 * s0:
                chosen = (dof2, Ac, tc)
                break
        if chosen is None:
            break
        dof2, A2, t2 = chosen
        n2, q2, d2, P2 = pair_up(A2, t2, fig, Q, treeQ, tol)
        improved = len(n2) > len(nodes_in) or (len(n2) == len(nodes_in) and (d2 ** 2).sum() < (dd ** 2).sum() - 1e-6)
        if accepted and not improved:
            break
        A, t, nodes_in, pts_in, dd, P, dof = A2, t2, n2, q2, d2, P2, dof2
        accepted = True
    if not accepted:
        nodes_in, pts_in, dd, P = pair_up(A, t, fig, Q, treeQ, tol)
    if dof == 3:
        sv = np.linalg.svd(A, compute_uv=False)
        dof = 2.0 + min(1.0, math.log(max(1.0, sv[0] / max(sv[1], 1e-9))) / math.log(p.id_max_aniso))
    score = llr_score(fig, nodes_in, pts_in, dd, P, tol, dof, Q, qmag, p, ctx)
    return Hypothesis(name=fig.name, score=score, n_in=len(nodes_in), n_nodes=len(fig.nodes), A=A, t=t,
                      in_nodes=nodes_in, in_pts=pts_in, tol=tol, resid=float(dd.mean()) if len(dd) else 0.0)


def naming_context(Q, p, dev, bright_xy=None):
    """Shared per-scene data: distance-to-nearest-point raster (for the soft screen) and the
    tree of the brightest stars (for the completeness penalty)."""
    cell = p.id_dt_cell
    G = int(math.ceil(SKY_W / cell)) + 2
    occupied = np.ones((G, G), np.uint8)
    q = np.clip((np.asarray(Q, float) / cell).astype(int), 0, G - 1)
    occupied[q[:, 1], q[:, 0]] = 0
    dist = cv2.distanceTransform(occupied, cv2.DIST_L2, 5).astype(np.float32) * cell
    dist_flat = torch.cat([torch.as_tensor(dist, device=dev).reshape(-1), torch.full((1,), 1e6, device=dev)])
    bright_tree = cKDTree(bright_xy) if bright_xy is not None and len(bright_xy) else cKDTree(Q)
    return dict(dist_flat=dist_flat, G=G, cell=cell, bright_tree=bright_tree)


def search_figure(fig, Q, qmag, treeQ, p, dev, ctx=None) -> Hypothesis:
    """Every seed node pair x every ordered pair of Q points x both handednesses gives a
    similarity; all are screened on the device with a Gaussian residual score read from the
    distance raster, and the best distinct ones are polished and scored exactly."""
    best = Hypothesis(name=fig.name, n_nodes=len(fig.nodes))
    n, m = len(fig.nodes), len(Q)
    if n < 3 or m < 3 or len(fig.pairs) == 0:
        return best
    ctx = ctx or naming_context(Q, p, dev)
    dist_flat, G, cell = ctx["dist_flat"], ctx["G"], ctx["cell"]
    OUTSIDE = G * G
    first, second = np.nonzero(~np.eye(m, dtype=bool))
    Qt = torch.as_tensor(Q, device=dev, dtype=torch.float32)
    first, second = torch.as_tensor(first, device=dev), torch.as_tensor(second, device=dev)
    seg = Qt[second] - Qt[first]
    seg_len, seg_ang, seg_start = torch.hypot(seg[:, 0], seg[:, 1]), torch.atan2(seg[:, 1], seg[:, 0]), Qt[first]
    n_seg = len(first)
    margin, off_m = p.id_shift_margin, p.id_off_margin
    max_off = max(p.id_max_off, int(p.id_max_off_frac * n))
    pairs = torch.as_tensor(fig.pairs, device=dev)
    count_hist = torch.zeros(n + 1, dtype=torch.int64, device=dev)
    shortlist = []                                  # rows: soft, count, flip, scale, angle, tx, ty
    for flip in (1.0, -1.0):
        F_nodes = torch.as_tensor(fig.nodes * np.array([1.0, flip]), device=dev, dtype=torch.float32)
        per_chunk = max(1, int(p.id_chunk_work // max(1, n_seg * n)))
        for c0 in range(0, len(pairs), per_chunk):
            pr = pairs[c0:c0 + per_chunk]
            na, nb = F_nodes[pr[:, 0]], F_nodes[pr[:, 1]]
            dn = nb - na
            node_len, node_ang = torch.hypot(dn[:, 0], dn[:, 1]), torch.atan2(dn[:, 1], dn[:, 0])
            s = seg_len[None, :] / node_len[:, None]
            ok = (s >= p.id_scale_lo) & (s <= p.id_scale_hi)
            th = seg_ang[None, :] - node_ang[:, None]
            c, si = torch.cos(th), torch.sin(th)
            tx = seg_start[None, :, 0] - s * (c * na[:, None, 0] - si * na[:, None, 1])
            ty = seg_start[None, :, 1] - s * (si * na[:, None, 0] + c * na[:, None, 1])
            ok &= (tx > -margin) & (tx < SKY_W + margin) & (ty > -margin) & (ty < SKY_H + margin)
            if not bool(ok.any()):
                continue
            s, th, tx, ty, c, si = s[ok], th[ok], tx[ok], ty[ok], c[ok], si[ok]
            X = tx[:, None] + s[:, None] * (c[:, None] * F_nodes[None, :, 0] - si[:, None] * F_nodes[None, :, 1])
            Y = ty[:, None] + s[:, None] * (si[:, None] * F_nodes[None, :, 0] + c[:, None] * F_nodes[None, :, 1])
            in_img = (X >= 0) & (X < SKY_W) & (Y >= 0) & (Y < SKY_H)
            n_off = ((X < -off_m) | (X > SKY_W + off_m) | (Y < -off_m) | (Y > SKY_H + off_m)).sum(1)
            cell_idx = (Y / cell).long().clamp(0, G - 1) * G + (X / cell).long().clamp(0, G - 1)
            cell_idx = torch.where(in_img, cell_idx, torch.full_like(cell_idx, OUTSIDE))
            d = dist_flat[cell_idx]
            tol = torch.clamp(p.id_tol_rel * s, min=p.id_tol_lo, max=p.id_tol_hi)
            count = (d <= (p.id_seed_mult * tol)[:, None]).sum(1)
            soft = torch.exp(-0.5 * (d / (p.id_soft_mult * tol)[:, None]) ** 2).sum(1)
            usable = (n_off <= max_off) & (count >= 3)
            count = torch.where(usable, count, torch.zeros_like(count))
            soft = torch.where(usable, soft, torch.zeros_like(soft))
            count_hist += torch.bincount(count, minlength=n + 1)[: n + 1]
            k = min(p.id_keep_per_chunk, len(soft))
            top = torch.topk(soft, k).indices
            shortlist.append(torch.stack([soft[top], count[top].float(), torch.full((k,), flip, device=dev),
                                          s[top], th[top], tx[top], ty[top]], 1))
    if not shortlist:
        return best
    hist = count_hist.cpu().numpy()
    nonzero = np.nonzero(hist[3:])[0]
    if len(nonzero) == 0:
        return best
    ties = int(hist[3 + nonzero[-1]])
    budget = p.id_verify_min if n <= 4 else max(p.id_verify_min, min(p.id_verify_max, ties))
    rows = torch.cat(shortlist)
    rows = rows[torch.argsort(-rows[:, 0])][: max(p.id_keep_per_chunk, 4 * budget)].cpu().numpy()
    seen = np.zeros((0, 4))
    checked = 0
    for soft0, count0, flip, s0, th0, tx0, ty0 in rows:
        if count0 < 3:
            continue
        th0 = (th0 + math.pi) % (2 * math.pi) - math.pi
        key = np.array([np.log(s0) * 20, th0, tx0 / 80.0, ty0 / 80.0])
        if len(seen):
            dth = np.abs((seen[:, 1] - th0 + math.pi) % (2 * math.pi) - math.pi) * 10
            if np.maximum(np.abs(seen[:, [0, 2, 3]] - key[[0, 2, 3]]).max(1), dth).min() < 1.0:
                continue
        seen = np.vstack([seen, key])
        if checked >= budget:
            break
        checked += 1
        c0, si0 = math.cos(th0), math.sin(th0)
        A = s0 * np.array([[c0, -si0], [si0, c0]]) @ np.diag([1.0, flip])
        h = polish(fig, A, np.array([tx0, ty0]), flip, Q, qmag, treeQ, p, ctx)
        if h.score > best.score:
            best = h
    best.n_checked = checked
    return best


def scrambled(fig, rng, p) -> Figure:
    """Null figure: each node rotated about the centroid by its own random angle."""
    ang = rng.uniform(0, 2 * math.pi, len(fig.nodes))
    r = np.hypot(fig.nodes[:, 0], fig.nodes[:, 1])
    norm = np.stack([r * np.cos(ang), r * np.sin(ang)], 1)
    return Figure(fig.name + "~", fig.raw, norm, fig.sizes, voting_pairs(norm, p), fig.shape)


def placement_fit(fig, hyp, Q, p):
    """Looser refit used only to place nodes on stars (scoring keeps the tight tolerance)."""
    if hyp.A is None or len(Q) < 3:
        return hyp.A, hyp.t
    tol = max(hyp.tol, p.id_tol_lo) * p.place_tol_mult
    flip = 1.0 if np.linalg.det(hyp.A) >= 0 else -1.0
    nodes_in, pts_in, _, _ = pair_up(hyp.A, hyp.t, fig, Q, cKDTree(Q), tol)
    if len(nodes_in) < max(3, hyp.n_in):
        return hyp.A, hyp.t
    return procrustes(fig.nodes[nodes_in], Q[pts_in], flip)


def name_constellation(Q, qmag, figs, p, bright_xy=None):
    """Rank all figures by calibrated score  max(LLR, 0) - null mean - null sd,  where the null
    comes from scrambled copies matched on the same points, pooled over similar node counts."""
    if not figs:
        return Hypothesis(), []
    if len(Q) < 3:
        biggest = max(figs, key=lambda f: len(f.nodes))
        return Hypothesis(name=biggest.name, score=0.0, n_nodes=len(biggest.nodes)), []
    treeQ = cKDTree(Q)
    ctx = naming_context(Q, p, DEVICE, bright_xy)
    ranked, null_scores = [], []
    for fi, fig in enumerate(figs):
        h = search_figure(fig, Q, qmag, treeQ, p, DEVICE, ctx)
        h.raw = max(float(h.score), 0.0)
        rng = np.random.default_rng(p.null_seed + 1000 * fi)
        null_scores.append([max(float(search_figure(scrambled(fig, rng, p), Q, qmag, treeQ, p, DEVICE, ctx).score), 0.0)
                            for _ in range(p.null_k)])
        ranked.append(h)
    if p.null_k > 0:
        n_nodes = np.array([len(f.nodes) for f in figs], float)
        own = np.array([np.mean(v) for v in null_scores])
        values = np.concatenate([np.asarray(v, float) for v in null_scores])
        for i, h in enumerate(ranked):
            w = np.exp(-0.5 * ((n_nodes - n_nodes[i]) / p.null_bw) ** 2)
            weights = np.concatenate([np.full(len(v), w[j]) for j, v in enumerate(null_scores)])
            mu = float((weights * values).sum() / max(weights.sum(), 1e-9))
            sd = math.sqrt(float((weights * (values - mu) ** 2).sum() / max(weights.sum(), 1e-9)))
            h.null_mean = (p.null_k * own[i] + p.null_shrink * mu) / (p.null_k + p.null_shrink)
            h.null_sd = sd + p.null_sd_floor
            h.score = h.raw - h.null_mean - h.null_sd
    ranked.sort(key=lambda h: -h.score)
    return ranked[0], ranked

## 6. Locating the tiles

**Ordinary tiles.** Each tile is resampled at 36 angles × 5 zooms, and every version is correlated with the whole sky. Peaks are picked with 5×5 max-pooling, and the best 1,500 become candidates. A second source adds the sky stars whose neighbour tables agree with the tile's descriptor (at least 2 matched neighbours).

**Verification.** Candidates are re-scored coarse to fine: a half-resolution screen, then an angle/zoom grid, then a sub-pixel shift grid. The correlation used here excludes the centre out to the tile's halo radius and removes the ring-average profile from both sides. A round halo therefore matches nothing, and only the asymmetric surroundings count.

For every tile we keep its best correlation, the best competitor more than 5 px away, and a z-score against 200 random poses.

**Bright-star tiles.** Here the whole-sky sweep is dominated by the halo. The candidates are the 500 brightest catalogue stars instead. The tile's own core centroid is placed on each one, and the score uses a full-disc correlation with every ring scaled to unit RMS, so the faint neighbours decide.

In [ ]:
class DeviceSky:
    """Stretched sky and neighbour tables on the torch device."""

    def __init__(self, field: StarField, sky_c, dev):
        T = lambda a: torch.as_tensor(a, device=dev)
        self.dev = dev
        self.N = field.N
        self.xy, self.flux = T(field.xy), T(field.flux)
        self.sat_area = T(getattr(field, "sat_area", np.zeros(field.N, np.float32)))
        self.nb_dx, self.nb_dy = T(field.nb_dx), T(field.nb_dy)
        self.nb_r = T(np.nan_to_num(field.nb_r, nan=1e6))
        self.nb_phi, self.nb_valid = T(field.nb_phi), T(field.nb_valid)
        self.nb_flux = T(np.nan_to_num(field.nb_flux, nan=1.0))
        self.img = T(sky_c)[None, None]
        self.H, self.W = sky_c.shape


def neighbour_votes(infos, dsky: DeviceSky, p):
    """For every (tile, sky star): how many tile neighbours land on sky neighbours under the
    best rotation/zoom implied by one of the tile's brightest neighbours.  Returns (count,
    score, zoom, angle) tensors of shape (tiles, stars)."""
    dev = dsky.dev
    B, K_t = len(infos), p.tile_max_neighbours
    r_t = np.ones((B, K_t), np.float32)
    th_t = np.zeros((B, K_t), np.float32)
    w_t = np.zeros((B, K_t), np.float32)
    f_t = np.ones((B, K_t), np.float32)
    e_t = np.zeros((B, K_t), np.float32)
    n_t = np.zeros(B, np.int64)
    for b, info in enumerate(infos):
        k = len(info.nb_r)
        n_t[b] = k
        if k:
            r_t[b, :k], th_t[b, :k], w_t[b, :k], f_t[b, :k], e_t[b, :k] = (
                info.nb_r, info.nb_theta, info.nb_weight, info.nb_flux, info.nb_edge)
    r_t, th_t, w_t, f_t, e_t, n_t = (torch.as_tensor(a, device=dev) for a in (r_t, th_t, w_t, f_t, e_t, n_t))
    N, K = dsky.nb_dx.shape
    best_sc = torch.full((B, N), -1.0, device=dev)
    best_cnt = torch.zeros((B, N), dtype=torch.int32, device=dev)
    best_s = torch.ones((B, N), device=dev)
    best_a = torch.zeros((B, N), device=dev)
    real = torch.arange(K_t, device=dev)[None, :] < n_t[:, None]
    n_seed = min(p.tile_seed_neighbours, int(n_t.max().item()) if B else 0)
    bytes_per_tile = N * K * K_t * K * 4 * 6
    group = max(1, int(p.mem_budget_gb * 1e9 // bytes_per_tile)) if dev.type == "cuda" else max(1, int(1.5e9 // bytes_per_tile))
    sky_dx = dsky.nb_dx[None, :, None, None, :]
    sky_dy = dsky.nb_dy[None, :, None, None, :]
    neg = lambda: torch.full((), -1.0, device=dev)
    for b0 in range(0, B, group):
        sl = slice(b0, min(B, b0 + group))
        b = sl.stop - sl.start
        for i in range(n_seed):
            has_seed = n_t[sl] > i
            if not bool(has_seed.any()):
                continue
            s = dsky.nb_r[None] / r_t[sl, i, None, None]
            ok = dsky.nb_valid[None] & (s >= p.zoom_lo) & (s <= p.zoom_hi) & has_seed[:, None, None]
            a = dsky.nb_phi[None] - th_t[sl, i, None, None]
            ang = a[..., None] + th_t[sl, None, None, :]
            rr = s[..., None] * r_t[sl, None, None, :]
            px, py = rr * torch.cos(ang), rr * torch.sin(ang)
            d2 = torch.square(px[..., None] - sky_dx)
            d2.add_(torch.square(py[..., None] - sky_dy))
            tol = p.nb_tol_px + p.nb_tol_rel * rr + p.nb_tol_edge * e_t[sl, None, None, :]
            hit = (d2 <= (tol * tol)[..., None]) & real[sl, None, None, :, None]
            tile_hit, sky_hit = hit.any(-1), hit.any(-2)
            cnt = torch.minimum(tile_hit.sum(-1), sky_hit.sum(-1))
            dmin, jm = d2.min(-1)
            del d2, hit
            quality = torch.where(tile_hit, 1.0 - 0.7 * torch.sqrt(dmin) / tol, torch.zeros((), device=dev))
            wsum = (quality * w_t[sl, None, None, :]).sum(-1)
            f_sky = dsky.nb_flux[None, :, None, :].expand(b, N, K, K).gather(3, jm)
            log_ratio = torch.log(f_t[sl, None, None, :] + 1e-6) - torch.log(f_sky + 1e-6)
            lr_hi = torch.where(tile_hit, log_ratio, torch.full((), -1e9, device=dev)).amax(-1)
            lr_lo = torch.where(tile_hit, log_ratio, torch.full((), 1e9, device=dev)).amin(-1)
            spread = torch.where(tile_hit.any(-1), (lr_hi - lr_lo).clamp(0, 3.0), torch.zeros((), device=dev))
            score = torch.where(ok, 0.6 * cnt.float() + wsum - p.flux_spread_w * spread, neg())
            sc_j, j = score.max(-1)
            better = sc_j > best_sc[sl]
            best_sc[sl] = torch.where(better, sc_j, best_sc[sl])
            best_cnt[sl] = torch.where(better, cnt.gather(2, j[..., None])[..., 0].int(), best_cnt[sl])
            best_s[sl] = torch.where(better, s.gather(2, j[..., None])[..., 0], best_s[sl])
            best_a[sl] = torch.where(better, a.gather(2, j[..., None])[..., 0], best_a[sl])
    return best_cnt, best_sc, best_s, best_a


_DEV_GRIDS = {}


def _dev_offsets(p, dev, step):
    key = (str(dev), step)
    if key not in _DEV_GRIDS:
        dx, dy, rad = tile_offsets(p.tile_centre)
        _DEV_GRIDS[key] = tuple(torch.as_tensor(a[::step, ::step], device=dev) for a in (dx, dy, rad))
    return _DEV_GRIDS[key]


def corr_many(dsky: DeviceSky, tile_c, cx, cy, zoom, ang, p, step=1, bg_inner=0.0, inner=None, whiten=None):
    """Correlation of one tile placed at many (centre, zoom, angle) poses.  The azimuthal mean
    profile is removed from both sides (so a round halo matches nothing), optionally every
    ring is scaled to unit RMS.  Also returns a background-only correlation and its pixel count."""
    dev = dsky.dev
    n = len(cx)
    empty = torch.zeros(0, device=dev)
    if n == 0:
        return empty, empty, empty
    U, V, rad = _dev_offsets(p, dev, step)
    tc = tile_c
    tile_stars = F.max_pool2d((tc.abs() > p.bgcorr_star_level).float()[None, None], 3, 1, 1)[0, 0] > 0
    if step > 1:
        tc, tile_stars = tc[::step, ::step], tile_stars[::step, ::step]
    L = U.shape[0]
    inner = p.corr_inner_r if inner is None else inner
    mask = ((rad <= p.corr_outer_r) & (rad >= inner)).float()
    mask_f = mask.reshape(-1)
    ring = torch.clamp(rad.round().long(), max=23).reshape(-1)
    N_RING = 24
    ring_n = torch.zeros(N_RING, device=dev).index_add_(0, ring, mask_f) + 1e-6

    def flatten_profile(W):
        sums = torch.zeros(W.shape[0], N_RING, device=dev).index_add_(1, ring, W)
        return (W - (sums / ring_n)[:, ring]) * mask_f

    whiten = p.whiten if whiten is None else whiten

    def equalise_rings(W):
        sq = torch.zeros(W.shape[0], N_RING, device=dev).index_add_(1, ring, W * W)
        return W / (torch.sqrt(sq / ring_n) + p.whiten_eps)[:, ring] * mask_f

    tm = tc * mask
    if p.corr_remove_profile:
        tm = flatten_profile(tm.reshape(1, -1)).reshape(L, L)
    if whiten:
        tm = equalise_rings(tm.reshape(1, -1)).reshape(L, L)
    T = (tm - (tm * mask).sum() / mask.sum()) * mask
    T_norm = torch.sqrt((T * T).sum()) + 1e-6
    bg_mask0 = (mask > 0) & (~tile_stars) & (rad >= max(p.bgcorr_inner_r, bg_inner))
    ncc, bgc, nbg = torch.empty(n, device=dev), torch.empty(n, device=dev), torch.empty(n, device=dev)
    Wmax, Hmax = float(dsky.W - 1), float(dsky.H - 1)
    batch = p.corr_batch if step > 1 else p.corr_batch // 4
    for b0 in range(0, n, batch):
        b1 = min(n, b0 + batch)
        m = b1 - b0
        co, sn, zz = torch.cos(ang[b0:b1]), torch.sin(ang[b0:b1]), zoom[b0:b1]
        X = cx[b0:b1, None, None] + zz[:, None, None] * (co[:, None, None] * U - sn[:, None, None] * V)
        Y = cy[b0:b1, None, None] + zz[:, None, None] * (sn[:, None, None] * U + co[:, None, None] * V)
        grid = torch.stack([X * (2.0 / Wmax) - 1.0, Y * (2.0 / Hmax) - 1.0], -1).reshape(1, m * L, L, 2)
        W = F.grid_sample(dsky.img, grid, mode="bilinear", padding_mode="zeros", align_corners=True).reshape(m, L, L)
        valid = ((X >= 0) & (X <= Wmax) & (Y >= 0) & (Y <= Hmax) & (mask > 0)).float()
        n_valid = valid.sum((1, 2)) + 1e-6
        Wm = W * valid
        if p.corr_remove_profile:
            Wm = flatten_profile(Wm.reshape(m, -1)).reshape(m, L, L) * valid
        if whiten:
            Wm = equalise_rings(Wm.reshape(m, -1)).reshape(m, L, L) * valid
        Wc = (Wm - (Wm.sum((1, 2)) / n_valid)[:, None, None]) * valid
        ncc[b0:b1] = (Wc * T[None]).sum((1, 2)) / (torch.sqrt((Wc * Wc).sum((1, 2))) * T_norm + 1e-6)
        sky_stars = F.max_pool2d((W.abs() > p.bgcorr_star_level).float()[:, None], 3, 1, 1)[:, 0] > 0
        bm = ((valid > 0) & bg_mask0[None] & (~sky_stars)).float()
        nb = bm.sum((1, 2)) + 1e-6
        Wb, Pb = Wm * bm, tc[None] * bm
        Wb = (Wb - (Wb.sum((1, 2)) / nb)[:, None, None]) * bm
        Pb = (Pb - (Pb.sum((1, 2)) / nb)[:, None, None]) * bm
        bgc[b0:b1] = (Wb * Pb).sum((1, 2)) / (torch.sqrt((Wb * Wb).sum((1, 2)) * (Pb * Pb).sum((1, 2))) + 1e-6)
        nbg[b0:b1] = nb
    return ncc, bgc, nbg


def best_on_grid(dsky, tile_c, cx, cy, zoom, ang, p, d_ang, d_zoom, d_xy, bg_inner, inner=None, whiten=None):
    """Evaluate every pose on a local (angle, zoom, shift) grid; return the best per pose as
    (score, ncc, bgc, nbg, zoom, angle, x, y)."""
    dev = dsky.dev
    n = len(cx)
    as_t = lambda a: torch.as_tensor(a, device=dev, dtype=torch.float32)
    d_ang, d_zoom, d_xy = as_t(d_ang), as_t(d_zoom), as_t(d_xy)
    ia, iz, ix = torch.meshgrid(torch.arange(len(d_ang), device=dev), torch.arange(len(d_zoom), device=dev),
                                torch.arange(len(d_xy), device=dev), indexing="ij")
    ia, iz, ix = ia.ravel(), iz.ravel(), ix.ravel()
    g = len(ia)
    AA = (ang[:, None] + d_ang[ia][None, :]).ravel()
    ZZ = (zoom[:, None] * d_zoom[iz][None, :]).ravel()
    XX = (cx[:, None] + d_xy[ix, 0][None, :]).ravel()
    YY = (cy[:, None] + d_xy[ix, 1][None, :]).ravel()
    v, bg, nb = corr_many(dsky, tile_c, XX, YY, ZZ, AA, p, bg_inner=bg_inner, inner=inner, whiten=whiten)
    v, bg, nb = v.view(n, g), bg.view(n, g), nb.view(n, g)
    sc = v + p.bgcorr_weight * bg
    k = sc.argmax(1)
    row = torch.arange(n, device=dev)
    return (sc[row, k], v[row, k], bg[row, k], nb[row, k], ZZ.view(n, g)[row, k], AA.view(n, g)[row, k],
            XX.view(n, g)[row, k], YY.view(n, g)[row, k])


@dataclass
class TileFix:
    x: float = -1.0
    y: float = -1.0
    ncc: float = -1.0
    ncc_alt: float = -1.0       # best competitor more than 5 px away
    z: float = 0.0              # (ncc - null median) / null MAD
    bgc: float = 0.0
    z_bgc: float = 0.0
    n_bgc: float = 0.0
    support: int = 0            # neighbour votes; negative = placed on a figure node
    n_nb: int = 0
    zoom: float = 1.0
    present: bool = False
    m: int = 0
    star: int = -1
    has_anchor: bool = False
    halo_r: float = 0.0
    mean: float = 0.0
    via_bright: bool = False
    n_pool: int = 0
    pool_margin: float = 0.0
    on_saturated: bool = False
    core_dx: float = 0.0
    core_dy: float = 0.0
    core_px: int = 0


def template_bank(tile_c, p, dev):
    """The tile resampled at every (angle, zoom) as zero-mean, unit-norm correlation kernels in
    search-image pixels, plus one disc mask per zoom."""
    stride = p.search_stride
    angles = np.arange(p.search_n_angles) * (2 * np.pi / p.search_n_angles)
    zooms = np.array(p.search_zooms, np.float32)
    r = int(math.ceil(p.corr_outer_r * zooms.max() / stride))
    u = torch.arange(-r, r + 1, device=dev, dtype=torch.float32) * stride
    X, Y = torch.meshgrid(u, u, indexing="xy")
    A = torch.as_tensor(angles, device=dev, dtype=torch.float32)
    Z = torch.as_tensor(zooms, device=dev)
    a, z = A.repeat_interleave(len(zooms)), Z.repeat(len(angles))
    ca, sa = torch.cos(a), torch.sin(a)
    px = (ca[:, None, None] * X + sa[:, None, None] * Y) / z[:, None, None] + p.tile_centre
    py = (-sa[:, None, None] * X + ca[:, None, None] * Y) / z[:, None, None] + p.tile_centre
    grid = torch.stack([px / 31.0 * 2 - 1, py / 31.0 * 2 - 1], -1)
    bank = F.grid_sample(tile_c[None, None].expand(len(a), 1, 32, 32), grid, mode="bilinear",
                         padding_mode="zeros", align_corners=True)[:, 0]
    rad = torch.hypot(X, Y)[None] / Z[:, None, None]
    masks = ((rad >= p.search_inner_r) & (rad <= p.corr_outer_r)).float()
    full = masks.repeat(len(angles), 1, 1)
    area = full.sum((1, 2))
    bank = (bank - (bank * full).sum((1, 2))[:, None, None] / area[:, None, None]) * full
    bank = bank / (torch.sqrt((bank * bank).sum((1, 2)))[:, None, None] + 1e-6)
    return bank[:, None], masks[:, None], a, z


class TorchSearchSky:
    """Search image for the GPU sweep, with cached local std maps."""

    def __init__(self, dsky, p):
        self.stride = p.search_stride
        self.img = F.avg_pool2d(dsky.img, self.stride) if self.stride > 1 else dsky.img
        self.img2 = self.img * self.img
        self._std = {}

    def window_std(self, masks):
        key = tuple(masks.shape)
        if key not in self._std:
            r = masks.shape[-1] // 2
            area = masks.sum((1, 2, 3))
            s1 = F.conv2d(self.img, masks, padding=r)
            s2 = F.conv2d(self.img2, masks, padding=r)
            self._std[key] = torch.sqrt((s2 - s1 * s1 / area[None, :, None, None]).clamp_min(1e-3))
        return self._std[key]


def sweep_torch(tile_c, sky: TorchSearchSky, p, dev):
    bank, masks, angles, zooms = template_bank(tile_c, p, dev)
    std = sky.window_std(masks)
    n_zoom = len(p.search_zooms)
    r = bank.shape[-1] // 2
    width = sky.img.shape[-1]
    k = p.search_peaks_each
    val, ys, xs, ang, zm = [], [], [], [], []
    for t0 in range(0, len(bank), p.search_batch):
        t1 = min(len(bank), t0 + p.search_batch)
        ncc = F.conv2d(sky.img, bank[t0:t1], padding=r)[0] / std[0, torch.arange(t0, t1, device=dev) % n_zoom]
        ncc = torch.where(ncc >= F.max_pool2d(ncc[:, None], 5, 1, 2)[:, 0], ncc, torch.full((), -1.0, device=dev))
        v, idx = torch.topk(ncc.reshape(t1 - t0, -1), k, dim=1)
        val.append(v.ravel())
        ys.append(torch.div(idx, width, rounding_mode="floor").ravel())
        xs.append((idx % width).ravel())
        ang.append(angles[t0:t1].repeat_interleave(k))
        zm.append(zooms[t0:t1].repeat_interleave(k))
    val, ys, xs, ang, zm = map(torch.cat, (val, ys, xs, ang, zm))
    keep = torch.topk(val, min(p.search_keep, len(val))).indices
    half_px = (sky.stride - 1) / 2.0
    return xs[keep].float() * sky.stride + half_px, ys[keep].float() * sky.stride + half_px, zm[keep], ang[keep], val[keep]


def tile_core(tile_c, p, raw=None):
    """Weighted centroid (tile px) and pixel count of the tile's central star core: the
    connected region above core_level of the peak excess, measured on the raw tile so a
    saturated plateau survives."""
    if raw is not None:
        img = cv2.GaussianBlur(np.asarray(raw, np.float32), (0, 0), 1.0)
    else:
        img = tile_c.detach().cpu().numpy() if torch.is_tensor(tile_c) else np.asarray(tile_c, np.float32)
    U, V, rad = tile_offsets(p.tile_centre)
    floor = float(np.median(img[rad >= 12]))
    iy, ix = np.unravel_index(int(np.argmax(np.where(rad <= p.core_search_r, img, -np.inf))), img.shape)
    top = float(img[iy, ix])
    if not np.isfinite(top) or top - floor <= 1e-3:
        return p.tile_centre, p.tile_centre, 0
    thr = floor + p.core_level * (top - floor)
    _, lab, _, _ = cv2.connectedComponentsWithStats((img >= thr).astype(np.uint8), connectivity=8)
    core = lab == lab[iy, ix]
    w = np.maximum(img - thr, 0)[core] + 1e-3
    return (float((w * U[core]).sum() / w.sum()) + p.tile_centre,
            float((w * V[core]).sum() / w.sum()) + p.tile_centre, int(core.sum()))


def scan_stars(tile_c, dsky, star_xy, star_sat, p, core, bg_inner=0.0, refine_n=None):
    """Put the tile's core on each given star, sweep angle/zoom, refine the best `refine_n`,
    then give the best few a coarse shift search (wider on saturated stars) and a sub-pixel
    polish.  Returns dict of per-star tensors v, x, y, s, a, bg, nbg, refined."""
    dev = dsky.dev
    n = len(star_xy)
    n_ang = p.node_n_angles
    zooms = torch.tensor(p.node_zooms, device=dev, dtype=torch.float32)
    per = n_ang * len(zooms)
    base = torch.arange(n_ang, device=dev, dtype=torch.float32) * (2 * math.pi / n_ang)
    A, Z = base.repeat_interleave(len(zooms)).repeat(n), zooms.repeat(n_ang * n)
    SX, SY = star_xy[:, 0].repeat_interleave(per), star_xy[:, 1].repeat_interleave(per)
    du, dv = float(core[0]) - p.tile_centre, float(core[1]) - p.tile_centre
    ca, sa = torch.cos(A), torch.sin(A)
    CX, CY = SX - Z * (ca * du - sa * dv), SY - Z * (sa * du + ca * dv)
    wh = p.whiten
    v = corr_many(dsky, tile_c, CX, CY, Z, A, p, inner=0.0, whiten=wh)[0].view(n, per)
    bv, bp = v.max(1)
    row = torch.arange(n, device=dev)
    out = dict(v=bv.clone(), x=CX.view(n, per)[row, bp].clone(), y=CY.view(n, per)[row, bp].clone(),
               s=Z.view(n, per)[row, bp].clone(), a=A.view(n, per)[row, bp].clone(),
               bg=torch.zeros(n, device=dev), nbg=torch.zeros(n, device=dev),
               refined=torch.zeros(n, dtype=torch.bool, device=dev))
    M = n if refine_n is None else min(int(refine_n), n)
    if M <= 0:
        return out
    top = torch.topk(bv, M).indices
    coarse_a = np.deg2rad(360.0 / n_ang / 2) * np.array([-1, -0.5, 0, 0.5, 1])
    coarse_z = 1.0 + 0.1 * np.array([-1, 0, 1])
    fine_a = np.deg2rad(p.fine_dang_deg) * np.array([-1, 0, 1])
    fine_z = 1.0 + p.fine_dzoom * np.array([-1, 0, 1])
    sub = p.fine_dxy * np.array([-2, -1, 0, 1, 2]) / 2.0
    fine_xy = np.array([(x, y) for x in sub for y in sub])

    def store(idx, res):
        out["v"][idx], out["bg"][idx], out["nbg"][idx] = res[1], res[2], res[3]
        out["s"][idx], out["a"][idx], out["x"][idx], out["y"][idx] = res[4], res[5], res[6], res[7]

    store(top, best_on_grid(dsky, tile_c, out["x"][top], out["y"][top], out["s"][top], out["a"][top], p,
                            coarse_a, coarse_z, np.zeros((1, 2)), bg_inner, 0.0, wh))
    out["refined"][top] = True
    top2 = top[torch.topk(out["v"][top], min(int(p.bright_jitter_n), M)).indices]
    sat_t = torch.as_tensor(np.asarray(star_sat, bool), device=dev)
    for grp, half_w, step in ((top2[sat_t[top2]], p.bright_jitter_sat_px, p.bright_jitter_sat_step),
                              (top2[~sat_t[top2]], p.bright_jitter_px, p.bright_jitter_step)):
        if len(grp) == 0:
            continue
        g = np.arange(-half_w, half_w + 1e-6, max(step, 0.25)) if half_w > 0 else np.zeros(1)
        shifts = np.array([(x, y) for x in g for y in g])
        rr = best_on_grid(dsky, tile_c, out["x"][grp], out["y"][grp], out["s"][grp], out["a"][grp], p,
                          fine_a, fine_z, shifts, bg_inner, 0.0, wh)
        store(grp, best_on_grid(dsky, tile_c, rr[6], rr[7], rr[4], rr[5], p, fine_a, fine_z, fine_xy, bg_inner, 0.0, wh))
    return out


def _null_z(dsky, tile_c, fix, p, gen, bg_inner, inner, whiten):
    """z-scores of the fix against the same tile placed on random stars at random poses."""
    dev = dsky.dev
    k = min(p.null_draws, dsky.N)
    pick = torch.randperm(dsky.N, generator=gen)[:k].to(dev)
    ra = torch.rand(k, generator=gen).to(dev) * 2 * math.pi
    rz = torch.exp(torch.rand(k, generator=gen).to(dev) * (math.log(1.25) - math.log(0.8)) + math.log(0.8))
    v0, b0, _ = corr_many(dsky, tile_c, dsky.xy[pick, 0], dsky.xy[pick, 1], rz, ra, p, bg_inner=bg_inner, inner=inner, whiten=whiten)
    med, bmed = float(v0.median()), float(b0.median())
    mad = float(1.4826 * (v0 - med).abs().median()) + 1e-3
    bmad = float(1.4826 * (b0 - bmed).abs().median()) + 1e-3
    fix.z = (fix.ncc - med) / mad
    fix.z_bgc = (fix.bgc - bmed) / bmad


def locate_bright(info, tile_c, dsky, votes, p, gen, fix):
    """Bright-star tiles: the whole-sky sweep is dominated by the halo, so the candidates are
    the brightest catalogue stars instead, scored with the ring-equalised full-disc correlation."""
    k = min(p.bright_pool, dsky.N)
    pool = torch.topk(dsky.flux, k).indices
    core = tile_core(tile_c, p, info.raw)
    fix.core_dx, fix.core_dy, fix.core_px = core[0] - p.tile_centre, core[1] - p.tile_centre, int(core[2])
    sat = (dsky.sat_area[pool] > 0).cpu().numpy()
    out = scan_stars(tile_c, dsky, dsky.xy[pool], sat, p, core, bg_inner=info.halo_r, refine_n=p.bright_refine_n)
    v = out["v"]
    b = int(v.argmax())
    fix.x, fix.y = float(out["x"][b]), float(out["y"][b])
    fix.ncc, fix.bgc, fix.n_bgc, fix.zoom = float(v[b]), float(out["bg"][b]), float(out["nbg"][b]), float(out["s"][b])
    fix.star, fix.on_saturated, fix.via_bright, fix.n_pool = int(pool[b]), bool(sat[b]), True, int(k)
    far = torch.hypot(out["x"] - fix.x, out["y"] - fix.y) > 5.0
    fix.ncc_alt = float(v[far].max()) if bool(far.any()) else float(v.median())
    fix.pool_margin = fix.ncc - fix.ncc_alt
    if votes is not None:
        fix.support = int(votes[0][pool[b]])
    _null_z(dsky, tile_c, fix, p, gen, info.halo_r, 0.0, p.whiten)
    return fix


def locate_tile(info, tile_c, dsky, search_sky, votes, p, gen):
    """Best sky position of one tile.  votes = (count, zoom, angle) rows for this tile or None."""
    dev = dsky.dev
    fix = TileFix(n_nb=len(info.nb_r), has_anchor=info.has_anchor, halo_r=float(info.halo_r), mean=float(info.mean))
    if dsky.N == 0:
        return fix
    if info.halo_r >= p.bright_radius and p.bright_pool > 0:
        return locate_bright(info, tile_c, dsky, votes, p, gen, fix)
    wh = p.whiten_dense
    if isinstance(search_sky, CvSearchSky):
        cx, cy, Z, A, _ = sweep_cv(tile_c, search_sky, p, dev)
    else:
        cx, cy, Z, A, _ = sweep_torch(tile_c, search_sky, p, dev)
    cnt = torch.zeros(len(cx), dtype=torch.int32, device=dev)
    if votes is not None and len(info.nb_r) >= 1:
        vcnt, vz, va = votes
        g = torch.nonzero(vcnt >= (2 if len(info.nb_r) >= 2 else 1))[:, 0]
        if len(g):
            cx, cy = torch.cat([cx, dsky.xy[g, 0]]), torch.cat([cy, dsky.xy[g, 1]])
            Z, A, cnt = torch.cat([Z, vz[g]]), torch.cat([A, va[g]]), torch.cat([cnt, vcnt[g]])
    inner = max(p.corr_inner_r, info.halo_r)
    v = corr_many(dsky, tile_c, cx, cy, Z, A, p, step=2, inner=inner, whiten=wh)[0]
    k0 = torch.topk(v, min(p.screen_keep, len(v))).indices
    cx, cy, Z, A, cnt = cx[k0], cy[k0], Z[k0], A[k0], cnt[k0]
    d_ang = np.deg2rad(p.coarse_dang_deg) * np.array([-1, -0.5, 0, 0.5, 1])
    d_zoom = 1.0 + p.coarse_dzoom * np.array([-1, 0, 1])
    rr = best_on_grid(dsky, tile_c, cx, cy, Z, A, p, d_ang, d_zoom, np.zeros((1, 2)), info.halo_r, inner, wh)
    v, Z, A, cx, cy = rr[0], rr[4], rr[5], rr[6], rr[7]
    kk = torch.topk(v, min(p.fine_keep, len(v))).indices
    d_ang = np.deg2rad(p.fine_dang_deg) * np.array([-1, 0, 1])
    d_zoom = 1.0 + p.fine_dzoom * np.array([-1, 0, 1])
    sub = p.fine_dxy * np.array([-2, -1, 0, 1, 2]) / 2.0
    d_xy = np.array([(x, y) for x in sub for y in sub])
    sc, vf, bgf, nbf, Zf, Af, xf, yf = best_on_grid(dsky, tile_c, cx[kk], cy[kk], Z[kk], A[kk], p, d_ang, d_zoom, d_xy,
                                                     info.halo_r, inner, wh)
    b = int(sc.argmax())
    fix.x, fix.y = float(xf[b]), float(yf[b])
    fix.ncc, fix.bgc, fix.n_bgc, fix.zoom = float(vf[b]), float(bgf[b]), float(nbf[b]), float(Zf[b])
    far = torch.hypot(xf - xf[b], yf - yf[b]) > 5.0
    fix.ncc_alt = float(vf[far].max()) if bool(far.any()) else float(vf.median())
    d2 = (dsky.xy[:, 0] - fix.x) ** 2 + (dsky.xy[:, 1] - fix.y) ** 2
    j = int(d2.argmin())
    fix.star = j
    if votes is not None and float(d2[j]) <= 9.0:
        fix.support = int(votes[0][j])
    _null_z(dsky, tile_c, fix, p, gen, info.halo_r, inner, wh)
    return fix


def is_present(f: TileFix, p) -> bool:
    """Present when: placed on a figure node; a bright-star tile (prior); correlation high
    enough alone; good and clearly ahead of the runner-up; far above its random-pose null;
    or moderately above it with enough neighbour votes."""
    if f.x < 0:
        return False
    if f.support < 0:
        return True
    if p.bright_always_present and f.halo_r >= p.bright_radius:
        return True
    rules = (
        f.halo_r >= p.bright_radius and f.ncc >= p.pres_bright_ncc,
        f.z_bgc >= p.pres_zbg and f.n_bgc >= 60,
        f.ncc >= p.pres_ncc_alone,
        f.ncc >= p.pres_ncc_floor and (f.ncc - f.ncc_alt) >= p.pres_margin,
        f.z >= p.pres_z_alone,
        f.support >= p.pres_min_support and f.z >= p.pres_z_weak,
    )
    return any(rules)


def nodes_to_stars(P, field, p):
    """Each in-image projected node -> the brightest catalogue star within node_snap_px."""
    xy, idx, flux = [], [], []
    for i, q in enumerate(P):
        if not (5 <= q[0] < SKY_W - 5 and 5 <= q[1] < SKY_H - 5):
            continue
        near = field.tree.query_ball_point(q, p.node_snap_px)
        if near:
            j = max(near, key=lambda s: field.flux[s])
            xy.append(field.xy[j]); idx.append(i); flux.append(field.flux[j])
    return np.array(xy, float).reshape(-1, 2), np.array(idx, int), np.array(flux, float)


def node_corr_table(infos, chosen, dsky, node_xy, p):
    """Ring-equalised correlation of every chosen tile at every node star."""
    dev = dsky.dev
    n_nodes = len(node_xy)
    if n_nodes == 0 or not chosen:
        return np.zeros((len(chosen), n_nodes))
    nxy = torch.as_tensor(np.asarray(node_xy, np.float32), device=dev)
    d2 = ((nxy[:, None, :] - dsky.xy[None, :, :]) ** 2).sum(-1)
    jn = d2.argmin(1)
    sat = ((dsky.sat_area[jn] > 0) & (d2[torch.arange(n_nodes, device=dev), jn] <= 25.0)).cpu().numpy()
    table = np.zeros((len(chosen), n_nodes))
    for row, k in enumerate(chosen):
        info = infos[k]
        tc = torch.as_tensor(info.img_c, device=dev)
        core = tile_core(tc, p, info.raw)
        table[row] = scan_stars(tc, dsky, nxy, sat, p, core, bg_inner=info.halo_r, refine_n=n_nodes)["v"].cpu().numpy()
    return table


def fill_nodes(fixes, infos, dsky, node_xy, node_flux, tiles, p):
    """Give undecided bright tiles the named figure's node stars, one each: tiles already on a
    free node keep it; the rest go greedily by correlation; blown-out tiles are matched to the
    brightest free nodes by brightness rank."""
    if len(node_xy) == 0:
        return []

    def undecided(f):
        if f.via_bright:
            return not (f.present and f.pool_margin >= p.bright_sure_margin)
        return not (f.present and (f.ncc >= p.pres_ncc_alone or (f.ncc - f.ncc_alt) >= p.dense_confident_margin))

    blown = [k for k, f in enumerate(fixes) if tiles[k].mean() >= p.blown_mean and undecided(f)]
    bright = [k for k, f in enumerate(fixes) if infos[k].halo_r >= p.bright_radius and undecided(f) and k not in blown]
    used, placed = set(), []
    if p.node_keep_px > 0:
        for k in sorted(bright + blown, key=lambda k: -fixes[k].ncc):
            f = fixes[k]
            if f.x < 0:
                continue
            d = np.hypot(node_xy[:, 0] - f.x, node_xy[:, 1] - f.y)
            j = int(np.argmin(d))
            if d[j] <= p.node_keep_px and j not in used:
                used.add(j)
                f.present, f.m, f.support = True, 1, -3
                placed.append((k, j, float(f.ncc)))
                (bright if k in bright else blown).remove(k)
    if bright:
        table = node_corr_table(infos, bright, dsky, node_xy, p)
        srt = np.sort(table, axis=1)
        margin = srt[:, -1] - srt[:, -2] if table.shape[1] > 1 else srt[:, -1]
        taken = set()
        for flat in np.argsort(-table, axis=None):
            i, j = np.unravel_index(flat, table.shape)
            if table[i, j] < p.node_min_ncc:
                break
            if i in taken or j in used or margin[i] < p.node_min_margin:
                continue
            if len(used) >= len(node_xy):
                break
            taken.add(i); used.add(j)
            f = fixes[bright[i]]
            f.x, f.y = float(node_xy[j, 0]), float(node_xy[j, 1])
            f.ncc, f.present, f.m, f.support = float(table[i, j]), True, 1, -1
            placed.append((bright[i], int(j), float(table[i, j])))
    if blown and p.assign_blown:
        free = [j for j in np.argsort(-node_flux) if j not in used]
        for k in sorted(blown, key=lambda k: -tiles[k].mean()):
            if not free:
                break
            j = free.pop(0)
            if node_flux[j] < p.blown_flux_frac * node_flux.max():
                break
            f = fixes[k]
            f.x, f.y = float(node_xy[j, 0]), float(node_xy[j, 1])
            f.present, f.m, f.support = True, 1, -2
            used.add(j)
            placed.append((k, int(j), -1.0))
    return placed

## 7. Presence and placement on the figure

**Presence** (`is_present`) says yes to any of the following:
- the tile was placed on a figure node;
- it is a bright-star tile, since correlation cannot separate present from absent for these, and most of them are present;
- correlation of at least 0.70;
- correlation of at least 0.55 that leads the runner-up by at least 0.08;
- a z-score of at least 8, or at least 3 with at least 3 neighbour votes.

The lead test is an additive form of Lowe's ratio test. The last rule is a double threshold: strong evidence stands alone, weak evidence needs support.

**Placement.** The winning transform is refitted with a 3× looser tolerance, used for placement only. Each projected node then snaps to the brightest catalogue star within 30 px. A bright tile is left alone only when it leads its runner-up by at least 0.08. Every other bright tile is handled in this order:
1. A tile already within 12 px of a free node keeps its position and claims that node.
2. The rest are assigned greedily, one per node, by their correlation at each node star.
3. Fully blown-out tiles (mean ≥ 150) cannot be correlated, so they take the brightest free nodes in brightness order.

A reported point on a figure star can only raise geometric recovery. Present points within 12 px of a node get `m = 1`.

## 8. Driver, metric and CSV

`solve_scene` chains the stages for one scene. `solve_many` prepares the next scenes on CPU threads while the current one runs, and writes a scene that fails as all-absent / `unknown`. `scene_metric` is the competition score. `save_csv` fills in `sample_submission.csv`. `evaluate_train` and `run_validation` are the two entry points.

In [ ]:
def read_scene(folder):
    images = glob.glob(os.path.join(folder, "*_image.png")) or glob.glob(os.path.join(folder, "*.png"))
    sky = cv2.imread(images[0], cv2.IMREAD_GRAYSCALE)
    tile_files = glob.glob(os.path.join(folder, "patches", "patch_*.png"))
    tile_files.sort(key=lambda f: int(re.findall(r"(\d+)", os.path.basename(f))[-1]))
    return sky, [cv2.imread(f, cv2.IMREAD_GRAYSCALE) for f in tile_files]


def preprocess_scene(sky, tiles, p):
    stars, sky_c, noise = extract_sky(sky, p)
    infos = [describe_tile(t, p) for t in tiles]
    reach = max([i.nb_r.max() for i in infos if len(i.nb_r)] + [10.0])
    R = float(reach * p.zoom_hi + 2.5)
    return dict(field=StarField(stars, p, R), sky_c=sky_c, noise=noise, infos=infos, R=R, tiles=tiles)


def naming_points(field, fixes, p):
    """Brightest catalogue stars plus the positions of tiles judged present, merged within
    id_merge_px; brightness = saturated-pixel count with flux as tie-break."""
    by_flux = np.argsort(-field.flux)
    top = by_flux[: p.id_n_stars]
    pts, flux, sat = [field.xy[top].astype(float)], [field.flux[top].astype(float)], [field.sat[top].astype(float)]
    present = [i for i, f in enumerate(fixes) if f.present]
    if present:
        pts.append(np.array([[fixes[i].x, fixes[i].y] for i in present]))
        flux.append(np.array([field.flux[fixes[i].star] for i in present], float))
        sat.append(np.array([field.sat[fixes[i].star] for i in present], float))
    Q, qflux, qsat = np.vstack(pts), np.concatenate(flux), np.concatenate(sat)
    keep = nms_points(Q, p.id_merge_px, qflux)
    qmag = qsat + (qflux / (qflux.max() + 1.0) if len(qflux) else qflux)
    bright = field.xy[by_flux[: p.id_n_complete]].astype(float)
    return Q, qflux, qsat, qmag, keep, bright


def solve_scene(prep, figs, p, tag="", fixes=None):
    t0 = time.time()
    field, sky_c, infos = prep["field"], prep["sky_c"], prep["infos"]
    dsky = DeviceSky(field, sky_c, DEVICE)
    say(p, f"[{tag}] sky stars={field.N} noise={prep['noise']:.2f}  ({time.time() - t0:.1f}s)")
    # 1. locate every tile
    if fixes is None:
        use_cv = DEVICE.type == "cpu" or getattr(p, "search_backend", "auto") == "cv"
        search_sky = CvSearchSky(dsky, p) if use_cv else TorchSearchSky(dsky, p)
        cnt, _, zz, aa = neighbour_votes(infos, dsky, p)
        gen = torch.Generator()
        gen.manual_seed(0)
        fixes = [locate_tile(info, torch.as_tensor(info.img_c, device=DEVICE), dsky, search_sky, (cnt[k], zz[k], aa[k]), p, gen)
                 for k, info in enumerate(infos)]
        del search_sky
    for f in fixes:
        f.present, f.m = is_present(f, p), 0
    snapshot = [copy.copy(f) for f in fixes]
    say(p, f"[{tag}] located {len(fixes)} tiles, present={sum(f.present for f in fixes)}  ({time.time() - t0:.1f}s)")
    # 2. name the constellation
    Q, qflux, qsat, qmag, keep, bright = naming_points(field, fixes, p)
    best, ranked = name_constellation(Q[keep], qmag[keep], figs, p, bright_xy=bright) if p.do_identify else (Hypothesis(), [])
    say(p, f"[{tag}] id={best.name} cal={best.score:.2f} raw={best.raw:.2f} null={best.null_mean:.1f}+{best.null_sd:.1f} "
           f"inl={best.n_in}/{best.n_nodes}" + (f"  runner-up {ranked[1].name} {ranked[1].score:.2f}" if len(ranked) > 1 else "")
           + f"  |Q|={int(keep.sum())}  ({time.time() - t0:.1f}s)")
    # 3. place undecided bright tiles on the figure, label members
    node_xy, node_flux = np.zeros((0, 2)), np.zeros(0)
    if best.A is not None:
        fig = next(f for f in figs if f.name == best.name)
        A, t = placement_fit(fig, best, Q[keep], p)
        node_xy, _, node_flux = nodes_to_stars(warp_pts(A, t, fig.nodes), field, p)
        placed = fill_nodes(fixes, infos, dsky, node_xy, node_flux, prep["tiles"], p)
        say(p, f"[{tag}] {len(node_xy)} node stars, {len(placed)} bright tiles placed  ({time.time() - t0:.1f}s)")
        for f in fixes:
            if f.present and len(node_xy) and np.hypot(node_xy[:, 0] - f.x, node_xy[:, 1] - f.y).min() <= p.member_px:
                f.m = 1
    best.extra = dict(fixes_before=snapshot, Q=Q[keep], qflux=qflux[keep], qsat=qsat[keep], node_xy=node_xy,
                      node_flux=node_flux, sky_xy=field.xy, sky_flux=field.flux, sky_sat=field.sat, noise=prep["noise"],
                      ranking=[(h.name, h.score, h.raw, h.null_mean, h.null_sd, h.n_in, h.n_nodes) for h in ranked])
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return fixes, best, ranked


def solve_many(jobs, figs, p):
    """jobs: (scene_id, folder).  CPU preprocessing of the next scenes overlaps with solving;
    a failing scene is emitted as all-absent / unknown."""
    results = []

    def prepare(job):
        sid, folder = job
        return sid, preprocess_scene(*read_scene(folder), p)

    with ThreadPoolExecutor(max_workers=max(1, p.prefetch)) as pool:
        pending = [pool.submit(prepare, j) for j in jobs[: p.prefetch]]
        nxt = p.prefetch
        for i in range(len(jobs)):
            sid = jobs[i][0]
            try:
                sid, prep = pending[i].result()
            except Exception:
                prep = None
                print(f"[{sid}] preparation FAILED: {traceback.format_exc()}", file=sys.stderr, flush=True)
            if nxt < len(jobs):
                pending.append(pool.submit(prepare, jobs[nxt]))
                nxt += 1
            if prep is None:
                results.append((sid, [], Hypothesis(), []))
                continue
            try:
                fixes, best, ranked = solve_scene(prep, figs, p, tag=sid)
            except Exception:
                print(f"[{sid}] scene FAILED, writing all-absent/unknown: {traceback.format_exc()}", file=sys.stderr, flush=True)
                fixes, best, ranked = [], Hypothesis(), []
            results.append((sid, fixes, best, ranked))
    return results



def ramp(d):
    return float(np.clip((36.0 - d) / 24.0, 0.0, 1.0))


def f1(tp, fp, fn):
    if tp + fp + fn == 0:
        return 1.0
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0


def scene_metric(truth, pred, true_name, pred_name):
    """truth / pred: lists of None or (x, y, m)."""
    both = [(t, q) for t, q in zip(truth, pred)]
    tp = sum(t is not None and q is not None for t, q in both)
    fp = sum(t is None and q is not None for t, q in both)
    fn = sum(t is not None and q is None for t, q in both)
    tn = len(truth) - tp - fp - fn
    presence = 0.5 * (f1(tp, fp, fn) + f1(tn, fn, fp))
    present = [(t, q) for t, q in both if t is not None]
    loc = sum(ramp(math.hypot(t[0] - q[0], t[1] - q[1])) for t, q in present if q is not None)
    loc = loc / len(present) if present else 1.0
    figure = [t for t in truth if t is not None and t[2] == 1]
    reported = [q for q in pred if q is not None]
    geo = 0.0
    if figure and reported:
        D = np.array([[math.hypot(a[0] - b[0], a[1] - b[1]) for b in reported] for a in figure])
        used_f, used_r = set(), set()
        for flat in np.argsort(D, axis=None):
            i, j = np.unravel_index(flat, D.shape)
            if i in used_f or j in used_r:
                continue
            used_f.add(i); used_r.add(j)
            geo += ramp(D[i, j])
    geo = geo / len(figure) if figure else 1.0
    ident = 1.0 if pred_name == true_name else 0.0
    return dict(presence=presence, localization=loc, geometric=geo, identification=ident,
                total=0.25 * presence + 0.20 * loc + 0.25 * geo + 0.30 * ident)


def parse_cell(cell):
    nums = re.findall(r"-?\d+(?:\.\d+)?", (cell or "").strip())
    if len(nums) < 2:
        return None
    x, y = float(nums[0]), float(nums[1])
    if x < 0 or y < 0:
        return None
    return (x, y, int(float(nums[2])) if len(nums) >= 3 else 0)


def read_truth(path):
    truth = {}
    with open(path, newline="") as fh:
        for row in csv.DictReader(fh):
            n = int(row["n_patches"])
            truth[row["Id"]] = ([parse_cell(row[f"patch_{i:02d}"]) for i in range(1, n + 1)], row["constellation"].strip())
    return truth


def to_cells(fixes):
    """Integer, image-clamped cells exactly as written to the CSV."""
    clamp = lambda v, hi: int(min(max(round(v), 0), hi - 1))
    return [(clamp(f.x, SKY_W), clamp(f.y, SKY_H), int(f.m)) if f.present else None for f in fixes]


def grid_tune(results, truth, p):
    """Optional: grid-search the presence thresholds on train (0.25 P + 0.20 L + 0.25 G).
    Prints the best setting; not used by default."""
    import itertools
    grid = dict(pres_ncc_alone=[0.7, 0.75, 0.8, 0.99], pres_ncc_floor=[0.55, 0.6, 0.65, 0.7],
                pres_margin=[0.03, 0.06, 0.08, 0.12], pres_z_alone=[6.5, 8.0, 99.0],
                pres_bright_ncc=[0.45, 0.55, 0.65, 99.0])
    best, best_v = None, -1.0
    for vals in itertools.product(*grid.values()):
        trial = tuned(p, **dict(zip(grid, vals)))
        parts = []
        for sid, fixes, _, _ in results:
            cells, name = truth[sid]
            pred = [(f.x, f.y, f.m) if is_present(f, trial) else None for f in fixes]
            s = scene_metric(cells, pred, name, "unknown")
            parts.append(0.25 * s["presence"] + 0.20 * s["localization"] + 0.25 * s["geometric"])
        if np.mean(parts) > best_v:
            best_v, best = float(np.mean(parts)), dict(zip(grid, vals))
    print(f"\n[tune] best presence thresholds (partial score {best_v:.3f}): " + " ".join(f"{k}={v}" for k, v in best.items()))
    return best


def save_csv(results, data_dir, out_path):
    with open(os.path.join(data_dir, "sample_submission.csv"), newline="") as fh:
        rows = list(csv.reader(fh))
    header, body = rows[0], rows[1:]
    by_id = {sid: (fixes, best) for sid, fixes, best, _ in results}
    with open(out_path, "w", newline="") as fh:
        out = csv.writer(fh, quoting=csv.QUOTE_MINIMAL)
        out.writerow(header)
        for row in body:
            sid, n = row[0], int(row[1])
            cells, name = [], "unknown"
            if sid in by_id:
                fixes, best = by_id[sid]
                cells, name = to_cells(fixes), (best.name or "unknown")
            line = [sid, str(n)]
            for i in range(len(header) - 3):
                c = cells[i] if i < len(cells) else None
                line.append("-1" if c is None else f"({int(round(c[0]))}, {int(round(c[1]))}, {int(c[2])})")
            out.writerow(line + [name])
    print("wrote", out_path)


def evaluate_train(data_dir, figs, p, show_tiles=True, tune=False, limit=0):
    """Run the labelled scenes and print the metric, the top-5 names and a per-tile table."""
    truth = read_truth(os.path.join(data_dir, "train_ground_truth.csv"))
    jobs = [(sid, os.path.join(data_dir, "train", sid)) for sid in truth][: (limit or None)]
    results = solve_many(jobs, figs, p)
    scores = []
    for sid, fixes, best, ranked in results:
        cells, true_name = truth[sid]
        s = scene_metric(cells, to_cells(fixes), true_name, best.name)
        scores.append(s)
        print(f"\n=== {sid}: true={true_name} pred={best.name}  " + "  ".join(f"{k}={v:.3f}" for k, v in s.items()))
        if ranked:
            print("    top-5 (calibrated; raw, null mean+sd): " + ", ".join(
                f"{h.name}({h.score:.2f}; {h.raw:.1f}, {h.null_mean:.1f}+{h.null_sd:.1f}, {h.n_in}/{h.n_nodes})" for h in ranked[:5]))
        true_h = next((h for h in ranked if h.name == true_name and h.A is not None), None)
        if true_h is not None:
            sv = np.linalg.svd(true_h.A, compute_uv=False)
            print(f"    true-figure fit: inliers={true_h.n_in}/{true_h.n_nodes} scale={math.sqrt(sv[0] * sv[1]):.0f}px/unit "
                  f"anisotropy={sv[0] / max(sv[1], 1e-9):.2f} resid={true_h.resid:.1f}px tol={true_h.tol:.1f}px")
        zooms = [f.zoom for f, t in zip(fixes, cells) if t is not None and f.present]
        if zooms:
            print(f"    recovered zooms: min={min(zooms):.2f} med={np.median(zooms):.2f} max={max(zooms):.2f}")
        if show_tiles:
            print("    tile  truth  pres  n_nb sup   ncc   alt      z    bgc  z_bg  nbg  zoom     err")
            for k, (f, t) in enumerate(zip(fixes, cells)):
                err = math.hypot(t[0] - f.x, t[1] - f.y) if (t is not None and f.x >= 0) else float("nan")
                kind = "abs" if t is None else ("fig" if t[2] == 1 else "off")
                print(f"    {k + 1:02d}     {kind}    {int(f.present)}    {f.n_nb}   {f.support}  {f.ncc:5.2f}  {f.ncc_alt:5.2f}  "
                      f"{f.z:5.1f}  {f.bgc:5.2f} {f.z_bgc:5.1f}  {f.n_bgc:3.0f}  {f.zoom:5.2f}  {err:7.1f}")
    print("\nMEAN: " + "  ".join(f"{k}={np.mean([s[k] for s in scores]):.3f}" for k in scores[0]))
    return results, (grid_tune(results, truth, p) if tune else None)


def run_validation(data_dir, figs, p, out_path="submission.csv", limit=0):
    with open(os.path.join(data_dir, "sample_submission.csv"), newline="") as fh:
        ids = [r[0] for r in list(csv.reader(fh))[1:]]
    jobs = [(sid, os.path.join(data_dir, "validation", sid)) for sid in ids][: (limit or None)]
    results = solve_many(jobs, figs, p)
    save_csv(results, data_dir, out_path)
    return results

In [ ]:
params = tuned(search_stride=1 if DEVICE.type == "cuda" else 2, mem_budget_gb=4.0, search_batch=6, corr_batch=16384)
figures = read_figures(DATA, params) if DATA else []
print("device:", DEVICE, "| search stride:", params.search_stride, "| figures:", len(figures))

## 9. Looking inside the pipeline (Pisces training scene)

These cells only draw pictures. They are skipped when the file runs as a script.

### Plot A: pixel distribution at three stages

Histogram and CDF of the raw sky, the background-subtracted sky, and the square-root-stretched image the correlation uses. The raw values bunch at the dark end. Background removal centres them on zero, and the stretch pulls the bright tail in without reordering intensities. Histogram equalisation would destroy the brightness ratios, so it is not used.

In [ ]:
if IN_NOTEBOOK:
    import time
    import matplotlib.pyplot as plt

    _t = time.time()
    _CACHE = globals().setdefault("_CACHE", {})

    def demo_scene(name="pisces"):
        """Read and preprocess one training scene once."""
        if name not in _CACHE:
            sky, tiles = read_scene(os.path.join(DATA, "train", name))
            _CACHE[name] = (sky, tiles, preprocess_scene(sky, tiles, params))
        return _CACHE[name]

    sky, tiles, prep = demo_scene()
    h, w = sky.shape
    coarse = cv2.resize(sky, (w // params.bg_shrink, h // params.bg_shrink), interpolation=cv2.INTER_AREA)
    bg = cv2.resize(cv2.medianBlur(coarse, params.bg_median_px | 1), (w, h), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    hp = sky.astype(np.float32) - bg
    stages = [("raw sky", sky[::3, ::3].ravel().astype(np.float32), (0, 255), 256),
              ("minus median background", hp[::3, ::3].ravel(), (-40, 200), 240),
              ("sqrt stretch (noise units)", prep["sky_c"][::3, ::3].ravel(), (-3, 8), 220)]
    fig, ax = plt.subplots(2, 3, figsize=(13, 6))
    for j, (title, v, rng, bins) in enumerate(stages):
        counts, edges = np.histogram(v, bins=bins, range=rng)
        mid = 0.5 * (edges[1:] + edges[:-1])
        ax[0, j].semilogy(mid, np.maximum(counts, 1), lw=1.2)
        ax[0, j].set_title(title, fontsize=10)
        ax[1, j].plot(mid, counts.cumsum() / max(counts.sum(), 1), lw=1.5)
        for q in (0.5, 0.99, 0.999):
            xq = float(np.percentile(v, 100 * q))
            ax[1, j].axvline(xq, color="k", ls=":", lw=0.8)
            ax[1, j].text(xq, q - 0.08, f"{q:g}: {xq:.1f}", fontsize=7, rotation=90, va="top")
        for a in ax[:, j]:
            a.set_xlim(rng); a.grid(True, alpha=0.3)
        ax[1, j].set_ylim(0, 1.02)
    ax[0, 0].set_ylabel("pixels (log)"); ax[1, 0].set_ylabel("CDF")
    inset = ax[0, 2].inset_axes([0.55, 0.5, 0.42, 0.45])
    xx = np.linspace(-5, 60, 200)
    inset.plot(xx, np.sign(xx) * np.sqrt(np.abs(xx)), color="C3", lw=1.2)
    inset.set_title("lookup: sign(x)·sqrt|x|", fontsize=7); inset.tick_params(labelsize=6)
    fig.suptitle(f"pisces: {(sky >= 250).mean() * 100:.3f}% saturated pixels, noise sigma {mad_sigma(hp[::7, ::7]):.1f}", fontsize=10)
    plt.tight_layout(); plt.show()
    print(f"plot A: {time.time() - _t:.1f}s")

### Plot B: matched filter against LoG and DoG

The same crop filtered three ways at the same σ: the Gaussian matched filter the pipeline uses, a difference of Gaussians, and a scale-normalised Laplacian of Gaussian. Green circles are the labelled present tiles and cyan circles the detections. The printout compares each detector's 350 brightest stars with the matched filter's.

In [ ]:
if IN_NOTEBOOK:
    _t = time.time()
    sky, tiles, prep = demo_scene()
    truth_cells, _ = read_truth(os.path.join(DATA, "train_ground_truth.csv"))["pisces"]
    labelled = np.array([c[:2] for c in truth_cells if c is not None], float)
    members = np.array([c[:2] for c in truth_cells if c is not None and c[2] == 1], float)
    h, w = sky.shape
    coarse = cv2.resize(sky, (w // params.bg_shrink, h // params.bg_shrink), interpolation=cv2.INTER_AREA)
    hp = sky.astype(np.float32) - cv2.resize(cv2.medianBlur(coarse, params.bg_median_px | 1), (w, h),
                                             interpolation=cv2.INTER_LINEAR).astype(np.float32)
    clean = cv2.medianBlur(hp, 3)
    sg = params.star_psf_sigma
    g1 = cv2.GaussianBlur(clean, (0, 0), sg)
    responses = {"matched filter (pipeline)": g1,
                 "DoG  G(1.2) - G(1.9)": g1 - cv2.GaussianBlur(clean, (0, 0), 1.6 * sg),
                 "LoG  -sigma^2 Laplacian": -sg * sg * cv2.Laplacian(g1, cv2.CV_32F, ksize=3)}
    detections = {}
    for name, r in responses.items():
        nz = mad_sigma(r[::7, ::7])
        detections[name] = (local_maxima(r, clean, params.star_k_sigma * nz, params.star_nms_radius, params.star_cap, border=3), nz)
    # 400 x 400 window holding the most labelled stars
    best = max(((int(((labelled[:, 0] >= x0) & (labelled[:, 0] < x0 + 400) & (labelled[:, 1] >= y0) & (labelled[:, 1] < y0 + 400)).sum()), x0, y0)
                for x0, y0 in ((int(np.clip(c[0] - 200, 0, w - 400)), int(np.clip(c[1] - 200, 0, h - 400))) for c in members)))
    _, x0, y0 = best
    win = (slice(y0, y0 + 400), slice(x0, x0 + 400))
    fig, ax = plt.subplots(1, 4, figsize=(14, 4.2))
    ax[0].imshow(np.sign(hp[win]) * np.sqrt(np.abs(hp[win])), cmap="gray", vmin=-2, vmax=12)
    ax[0].set_title("background-free crop", fontsize=8)
    for a, (name, r) in zip(ax[1:], responses.items()):
        st, nz = detections[name]
        a.imshow(r[win] / nz, cmap="magma", vmin=-2, vmax=15)
        inside = (st["x"] >= x0) & (st["x"] < x0 + 400) & (st["y"] >= y0) & (st["y"] < y0 + 400)
        a.scatter(st["x"][inside] - x0, st["y"][inside] - y0, s=26, facecolors="none", edgecolors="cyan", lw=0.7)
        a.set_title(f"{name}\n{int(inside.sum())} in crop, {len(st['x'])} in sky", fontsize=8)
    for a in ax:
        a.scatter(labelled[:, 0] - x0, labelled[:, 1] - y0, s=120, facecolors="none", edgecolors="lime", lw=1.2)
        a.set_xlim(0, 400); a.set_ylim(400, 0); a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.show()
    ref = detections["matched filter (pipeline)"][0]
    ref_tree = cKDTree(np.stack([ref["x"], ref["y"]], 1))
    ref_top = np.argsort(-ref["flux"])[:350]
    for name, (st, _) in detections.items():
        xy = np.stack([st["x"], st["y"]], 1)
        d, j = ref_tree.query(xy[np.argsort(-st["flux"])[:350]])
        shared = np.isin(j[d <= 2.0], ref_top).sum()
        worst = cKDTree(xy).query(members)[0].max()
        print(f"{name:28s} N={len(xy):6d}  top-350 shared with matched filter {shared / 3.5:5.1f}%  worst figure-star distance {worst:.2f}px")
    print(f"plot B: {time.time() - _t:.1f}s")

### Plot C: why plain `TM_CCOEFF_NORMED` fails on a bright tile

The textbook recipe (`cv2.matchTemplate` + `minMaxLoc` over a rotation sweep) is run on a bright figure tile. Its maximum lands on the wrong star, because every bright star has the same round halo. The pipeline's correlation, at the truth and at that false peak, is then compared with its own random-pose null. The ground truth is used only to draw the plot.

In [ ]:
if IN_NOTEBOOK:
    _t = time.time()
    sky, tiles, prep = demo_scene()
    infos, sky_c = prep["infos"], prep["sky_c"]
    truth_cells, _ = read_truth(os.path.join(DATA, "train_ground_truth.csv"))["pisces"]

    def textbook_match(image, tile, crop=22):
        best = None
        for a in range(0, 360, 10):
            rot = cv2.warpAffine(tile, cv2.getRotationMatrix2D((15.5, 15.5), a, 1.0), (32, 32),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)[5:5 + crop, 5:5 + crop]
            res = cv2.matchTemplate(image, np.ascontiguousarray(rot), cv2.TM_CCOEFF_NORMED)
            _, mv, _, ml = cv2.minMaxLoc(res)
            if best is None or mv > best[0]:
                best = (mv, (ml[0] + crop // 2, ml[1] + crop // 2), a, res)
        return best

    bright_members = sorted((k for k, (i, t) in enumerate(zip(infos, truth_cells))
                             if t is not None and t[2] == 1 and i.halo_r >= params.bright_radius), key=lambda k: -infos[k].halo_r)
    for k in bright_members[:3]:
        mv, loc, ang, res = textbook_match(sky_c, infos[k].img_c)
        err = math.hypot(loc[0] - truth_cells[k][0], loc[1] - truth_cells[k][1])
        if err > 36:
            break
    if err <= 36:
        print("note: the textbook recipe happened to find this tile")
    tx, ty = float(truth_cells[k][0]), float(truth_cells[k][1])
    at_truth = float(res[int(np.clip(round(ty) - 11, 0, res.shape[0] - 1)), int(np.clip(round(tx) - 11, 0, res.shape[1] - 1))])
    dsky = DeviceSky(prep["field"], sky_c, DEVICE)
    tc = torch.as_tensor(infos[k].img_c, device=DEVICE)
    zooms = np.array(params.node_zooms, np.float32)
    A = torch.as_tensor(np.repeat(np.arange(36) * 2 * np.pi / 36, len(zooms)), dtype=torch.float32, device=DEVICE)
    Z = torch.as_tensor(np.tile(zooms, 36), dtype=torch.float32, device=DEVICE)
    g = torch.Generator(); g.manual_seed(0)
    n_null = min(params.null_draws, dsky.N)
    pick = torch.randperm(dsky.N, generator=g)[:n_null]
    ra = torch.rand(n_null, generator=g) * 2 * math.pi
    rz = torch.exp(torch.rand(n_null, generator=g) * (math.log(1.25) - math.log(0.8)) + math.log(0.8))
    variants = {"ring-equalised full disc (bright path)": dict(whiten=True, inner=0.0),
                "halo-free annulus (dense path)": dict(whiten=False, inner=max(params.corr_inner_r, infos[k].halo_r))}
    rows = {}
    for label, kw in variants.items():
        at = lambda x, y: float(corr_many(dsky, tc, torch.full_like(A, float(x)), torch.full_like(A, float(y)), Z, A, params,
                                          bg_inner=infos[k].halo_r, **kw)[0].max())
        null = corr_many(dsky, tc, dsky.xy[pick, 0], dsky.xy[pick, 1], rz, ra, params, bg_inner=infos[k].halo_r, **kw)[0].numpy()
        rows[label] = (at(tx, ty), at(*loc), null)
    fig = plt.figure(figsize=(15, 4.8))
    gs = fig.add_gridspec(1, 4, width_ratios=[0.8, 1.6, 1.4, 1.4])
    a0 = fig.add_subplot(gs[0]); a0.imshow(tiles[k], cmap="gray"); a0.set_xticks([]); a0.set_yticks([])
    a0.set_title(f"patch_{k + 1:02d}, halo {infos[k].halo_r:.0f}px", fontsize=9)
    a1 = fig.add_subplot(gs[1])
    a1.imshow(cv2.resize(res, (res.shape[1] // 4, res.shape[0] // 4), interpolation=cv2.INTER_AREA), cmap="viridis",
              extent=[11, 11 + res.shape[1], 11 + res.shape[0], 11], vmin=-0.2, vmax=1)
    a1.scatter([tx], [ty], s=160, facecolors="none", edgecolors="lime", lw=1.5, label=f"truth: {at_truth:.2f}")
    a1.scatter([loc[0]], [loc[1]], s=160, marker="s", facecolors="none", edgecolors="red", lw=1.5, label=f"peak: {mv:.2f} ({err:.0f}px off)")
    a1.legend(fontsize=7, loc="lower left"); a1.set_title(f"TM_CCOEFF_NORMED, best of 36 angles ({ang} deg)", fontsize=9)
    for spec, (label, (v_t, v_f, null)) in zip((gs[2], gs[3]), rows.items()):
        a = fig.add_subplot(spec)
        a.hist(null, bins=30, color="0.75", label="200 random poses")
        a.axvline(v_t, color="lime", lw=2, label=f"truth {v_t:.2f}")
        a.axvline(v_f, color="red", lw=2, ls="--", label=f"false peak {v_f:.2f}")
        a.set_title(label, fontsize=9); a.legend(fontsize=7)
    plt.tight_layout(); plt.show()
    for label, (v_t, v_f, null) in rows.items():
        med = np.median(null); mad = 1.4826 * np.median(np.abs(null - med)) + 1e-3
        print(f"{label:40s} truth {v_t:.3f}  false {v_f:.3f}  null max {null.max():.3f}  z(truth) {(v_t - med) / mad:.1f}")
    print(f"plot C: {time.time() - _t:.1f}s")

### Plot D: presence evidence and the decision rule

The values shown are those before naming and placement. The left panel shows best vs runner-up correlation, the middle panel best correlation vs z-score, both for ordinary tiles, and the right panel shows bright-star tiles. The lines are the thresholds of `is_present`. The plot uses `train_results` from section 10 if it has run, and otherwise locates the Pisces tiles live with naming switched off.

In [ ]:
if IN_NOTEBOOK:
    _t = time.time()
    if globals().get("train_results"):
        source, runs = "evaluate_train", train_results
    else:
        print("no train results yet: locating pisces now (naming off)")
        sky, tiles, prep = demo_scene()
        fx, hyp, _ = solve_scene(prep, [], tuned(params, do_identify=False, verbose=0), tag="pisces")
        source, runs = "live pisces run", [("pisces", fx, hyp, [])]
    truth = read_truth(os.path.join(DATA, "train_ground_truth.csv"))
    rows = []
    for sid, fixes, hyp, _ in runs:
        before = getattr(hyp, "extra", {}).get("fixes_before") or fixes
        for f, t in zip(before, truth[sid][0]):
            kind = "absent" if t is None else ("figure" if t[2] == 1 else "off-figure")
            rows.append(dict(kind=kind, ncc=f.ncc, alt=f.ncc_alt, z=f.z, support=f.support,
                             bright=f.halo_r >= params.bright_radius, said=is_present(f, params), truth=t is not None))
    colour = {"figure": "tab:green", "off-figure": "tab:blue", "absent": "tab:red"}
    fig, ax = plt.subplots(1, 3, figsize=(16, 5))
    for kind, col in colour.items():
        marker = "x" if kind == "absent" else "o"
        plain = [r for r in rows if r["kind"] == kind and not r["bright"]]
        bright = [r for r in rows if r["kind"] == kind and r["bright"]]
        ax[0].scatter([r["ncc"] for r in plain], [r["alt"] for r in plain], c=col, marker=marker, s=28, alpha=0.85, label=f"{kind} ({len(plain)})")
        ax[1].scatter([r["ncc"] for r in plain], [r["z"] for r in plain], c=col, marker=marker, s=28, alpha=0.85)
        ax[2].scatter([r["ncc"] for r in bright], [r["alt"] for r in bright], c=col, marker=marker, s=28, alpha=0.85, label=f"{kind} ({len(bright)})")
    xs = np.linspace(params.pres_ncc_floor, params.pres_ncc_alone, 20)
    ax[0].axvline(params.pres_ncc_alone, color="k", lw=1, label=f"ncc >= {params.pres_ncc_alone}")
    ax[0].plot(xs, xs - params.pres_margin, "k--", lw=1, label=f"lead >= {params.pres_margin}")
    ax[0].axvline(params.pres_ncc_floor, color="k", ls=":", lw=0.8, label=f"ncc >= {params.pres_ncc_floor}")
    ax[1].axhline(params.pres_z_alone, color="k", lw=1, label=f"z >= {params.pres_z_alone}")
    ax[1].axhline(params.pres_z_weak, color="k", ls="--", lw=1, label=f"z >= {params.pres_z_weak} with >= {params.pres_min_support} votes")
    voted = [r for r in rows if not r["bright"] and r["support"] >= params.pres_min_support]
    ax[1].scatter([r["ncc"] for r in voted], [r["z"] for r in voted], s=110, facecolors="none", edgecolors="k", lw=0.8, label="enough votes")
    for a, xl, yl, title in ((ax[0], "best ncc", "runner-up ncc (> 5 px away)", "ordinary tiles: lead test"),
                             (ax[1], "best ncc", "z vs random poses", "ordinary tiles: null test"),
                             (ax[2], "best candidate-star ncc", "runner-up star", "bright tiles: present by prior")):
        a.set_xlabel(xl); a.set_ylabel(yl); a.set_title(title, fontsize=10); a.legend(fontsize=7, loc="upper left"); a.set_xlim(0, 1)
    for a in (ax[0], ax[2]):
        a.plot([0, 1], [0, 1], color="0.6", lw=0.6); a.set_ylim(0, 1)
    fig.suptitle(f"{len(rows)} training tiles before naming (source: {source})", fontsize=10)
    plt.tight_layout(); plt.show()
    for flag in (False, True):
        sub = [r for r in rows if r["bright"] == flag]
        tp = sum(r["said"] and r["truth"] for r in sub); fp = sum(r["said"] and not r["truth"] for r in sub)
        fn = sum(not r["said"] and r["truth"] for r in sub)
        print(f"{'bright' if flag else 'ordinary'} tiles: {len(sub):3d}  TP {tp:3d}  FP {fp:3d}  FN {fn:3d}  TN {len(sub) - tp - fp - fn:3d}")
    print(f"plot D: {time.time() - _t:.1f}s")

### Plot E: the named figure on the sky

The winning figure is projected onto the Pisces sky. Its matched nodes are joined to their stars, and nodes without a star are circled in red. Next to it are the drawing the nodes came from and the calibrated top-5 (`raw − null mean − null sd`).

In [ ]:
if IN_NOTEBOOK:
    _t = time.time()
    hyp = next((h for sid, _, h, _ in (globals().get("train_results") or []) if sid == "pisces" and h.A is not None), None)
    if hyp is None:
        print("no train results yet: naming pisces from its bright stars now (a few minutes on CPU)")
        sky, tiles, prep = demo_scene()
        Q, qflux, qsat, qmag, keep, bright = naming_points(prep["field"], [], params)
        hyp, ranked = name_constellation(Q[keep], qmag[keep], figures, params, bright_xy=bright)
        hyp.extra = dict(Q=Q[keep], ranking=[(h.name, h.score, h.raw, h.null_mean, h.null_sd, h.n_in, h.n_nodes) for h in ranked])
    fig_obj = next(f for f in figures if f.name == hyp.name)
    Q = np.asarray(hyp.extra["Q"], float)
    scale = math.sqrt(abs(np.linalg.det(hyp.A)))
    tol = min(params.id_tol_hi, max(params.id_tol_lo, params.id_tol_rel * scale))
    n_in, p_in, dd, P = pair_up(hyp.A, hyp.t, fig_obj, Q, cKDTree(Q), tol)
    sky, _ = read_scene(os.path.join(DATA, "train", "pisces"))
    small = cv2.resize(sky, (sky.shape[1] // 4, sky.shape[0] // 4), interpolation=cv2.INTER_AREA)
    fig = plt.figure(figsize=(16, 6))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.5, 1.0, 1.1])
    a = fig.add_subplot(gs[0])
    a.imshow(np.sqrt(small / 255.0), cmap="gray", extent=[0, SKY_W, SKY_H, 0], vmin=0, vmax=1)
    a.scatter(Q[:, 0], Q[:, 1], s=4, c="0.7", label=f"{len(Q)} naming points")
    in_img = (P[:, 0] >= 0) & (P[:, 0] < SKY_W) & (P[:, 1] >= 0) & (P[:, 1] < SKY_H)
    lost = np.ones(len(P), bool); lost[n_in] = False
    a.scatter(P[n_in, 0], P[n_in, 1], s=90, facecolors="none", edgecolors="lime", lw=1.5, label=f"{len(n_in)}/{len(P)} nodes on a star")
    a.scatter(P[lost & in_img, 0], P[lost & in_img, 1], s=90, facecolors="none", edgecolors="red", lw=1.2,
              label=f"{int((lost & in_img).sum())} without a star, {int((~in_img).sum())} off-image")
    for i, j in zip(n_in, p_in):
        a.plot([P[i, 0], Q[j, 0]], [P[i, 1], Q[j, 1]], color="lime", lw=0.8)
    a.set_xlim(-50, SKY_W + 50); a.set_ylim(SKY_H + 50, -50); a.legend(fontsize=7, loc="lower left")
    a.set_title(f"named '{hyp.name}', {scale:.0f} px/unit, mean residual {dd.mean() if len(dd) else 0:.1f}px", fontsize=9)
    a = fig.add_subplot(gs[1])
    art = cv2.imread(os.path.join(DATA, "patterns", f"{fig_obj.name}_pattern.png"), cv2.IMREAD_UNCHANGED)
    if art.ndim == 3 and art.shape[2] == 4:
        alpha = art[:, :, 3:4].astype(float) / 255
        art = (art[:, :, :3] * alpha + 40 * (1 - alpha)).astype(np.uint8)
    a.imshow(cv2.cvtColor(art, cv2.COLOR_BGR2RGB))
    a.scatter(fig_obj.raw[n_in, 0], fig_obj.raw[n_in, 1], s=70, facecolors="none", edgecolors="lime", lw=1.5)
    a.scatter(fig_obj.raw[lost, 0], fig_obj.raw[lost, 1], s=70, facecolors="none", edgecolors="red", lw=1.2)
    a.set_title(f"{fig_obj.name}: {len(fig_obj.nodes)} nodes, {len(fig_obj.pairs)} seed pairs", fontsize=9)
    a.set_xticks([]); a.set_yticks([])
    a = fig.add_subplot(gs[2])
    top = hyp.extra["ranking"][:5][::-1]
    a.barh([r[0] for r in top], [r[1] for r in top], color=["tab:green" if r[0] == hyp.name else "0.6" for r in top])
    for y, r in enumerate(top):
        a.text(max(r[1], 0) + 0.3, y, f"raw {r[2]:.1f} - null {r[3]:.1f}+{r[4]:.1f}  {r[5]}/{r[6]}", va="center", fontsize=7)
    a.set_xlim(0, max(r[1] for r in top) * 1.6 + 1); a.set_xlabel("calibrated score"); a.set_title("top 5", fontsize=9)
    plt.tight_layout(); plt.show()
    print(f"plot E: {time.time() - _t:.1f}s")

## 10. Training scenes

This runs the three labelled scenes and prints the metric components, the top-5 names with their raw and null scores, the true figure's fit, and a per-tile table. `evaluate_train(..., tune=True)` also grid-searches the presence thresholds. It is off by default, because 116 labelled tiles are too few to tune on safely.

In [ ]:
if IN_NOTEBOOK:
    train_results, _ = evaluate_train(DATA, figures, params, show_tiles=True, tune=False)

## 11. Validation scenes and the submission

Scenes run in the order of `sample_submission.csv`, and the CSV keeps its layout (`-1` or `"(x, y, m)"`, then the name). A wrong name costs no more than `unknown`, so a name is always given. The submitted run uses search stride 2, which quarters the sweep cost for a small recall loss.

In [ ]:
params = tuned(search_stride=2, mem_budget_gb=4.0, search_batch=6, corr_batch=16384)
figures = read_figures(DATA, params)
print("device:", DEVICE, "| search stride:", params.search_stride, "| figures:", len(figures))

In [ ]:
OUT_DIR = "/content" if IN_COLAB else "."
if IN_NOTEBOOK:
    import pandas as pd
    val_results = run_validation(DATA, figures, params, out_path=os.path.join(OUT_DIR, "submission.csv"))
    sub = pd.read_csv(os.path.join(OUT_DIR, "submission.csv"))
    print(sub.shape)
    print(sub["constellation"].value_counts())
    if IN_COLAB:
        from google.colab import files
        files.download(os.path.join(OUT_DIR, "submission.csv"))

## 12. Follow-up experiments on the validation run

Each of the three cells below applies one general rule to every scene. None refers to a particular scene.

1. **Naming sensitivity.** Scenes whose name is weak (lead under 7 or score under 3) are re-named with 300 and with 500 bright stars, reusing the saved localisation. This is diagnostic only.
2. **Adaptive full resolution.** Weak scenes are re-run at stride 1. The new result is kept if it agrees with the old name, or if it changes the name with a lead at least 2 higher and a score at least as high.
3. **Wider candidate recall.** A full re-run keeping 8 peaks per template, 1,800 candidates and a 180-candidate screen.

In [ ]:
from types import SimpleNamespace


def lead(ranked):
    return ranked[0].score - ranked[1].score if len(ranked) >= 2 else 999.0


def rename_from_saved(hyp, p2):
    """Re-run naming on the localisation saved in hyp.extra, with different settings."""
    ex = hyp.extra
    stars = SimpleNamespace(xy=np.asarray(ex["sky_xy"]), flux=np.asarray(ex["sky_flux"]), sat=np.asarray(ex["sky_sat"]))
    Q, qflux, qsat, qmag, keep, bright = naming_points(stars, ex["fixes_before"], p2)
    return name_constellation(Q[keep], qmag[keep], figures, p2, bright_xy=bright)


if IN_NOTEBOOK:
    fewer, more = tuned(params, id_n_stars=300), tuned(params, id_n_stars=500, id_n_complete=800)
    weak = []
    print("=" * 80)
    for sid, fixes, hyp, ranked in val_results:
        g = lead(ranked)
        print(f"{sid:18s} {hyp.name:20s} score={hyp.score:7.2f} lead={g:6.2f} runner={ranked[1].name if len(ranked) > 1 else '-'}")
        if g < 7.0 or hyp.score < 3.0:
            weak.append((sid, fixes, hyp, ranked))
    print("weak:", [w[0] for w in weak])
    NAME_SWEEP, _t = {}, time.time()
    for n, (sid, fixes, hyp, ranked) in enumerate(weak, 1):
        print(f"\n[{n}/{len(weak)}] {sid}\n  default: {hyp.name:20s} score={hyp.score:7.2f} lead={lead(ranked):6.2f}")
        NAME_SWEEP[sid] = {"default": (hyp, ranked, lead(ranked))}
        for label, p2 in (("300 stars", fewer), ("500 stars", more)):
            h2, r2 = rename_from_saved(hyp, p2)
            NAME_SWEEP[sid][label] = (h2, r2, lead(r2))
            print(f"  {label}: {h2.name:20s} score={h2.score:7.2f} lead={lead(r2):6.2f}" + (f" runner={r2[1].name}" if len(r2) > 1 else ""))
    print(f"\nnaming sweep: {(time.time() - _t) / 60:.2f} min")

In [ ]:
if IN_NOTEBOOK:
    full = tuned(params, search_stride=1, mem_budget_gb=4.0, search_batch=6, corr_batch=16384)
    figures_full = read_figures(DATA, full)
    adaptive, _t = [], time.time()
    for sid, fixes, hyp, ranked in val_results:
        g = lead(ranked)
        if not (g < 7.0 or hyp.score < 3.0):
            print(f"[{sid}] keep stride-2 result: {hyp.name} score={hyp.score:.2f} lead={g:.2f}")
            adaptive.append((sid, fixes, hyp, ranked))
            continue
        print(f"\n[{sid}] re-running at full resolution (was {hyp.name}, score={hyp.score:.2f}, lead={g:.2f})")
        st = time.time()
        fx2, h2, r2 = solve_scene(preprocess_scene(*read_scene(os.path.join(DATA, "validation", sid)), full), figures_full, full, tag=sid + "_full")
        g2 = lead(r2)
        print(f"[{sid}] {hyp.name} ({hyp.score:.2f}, lead {g:.2f}) -> {h2.name} ({h2.score:.2f}, lead {g2:.2f}) | {(time.time() - st) / 60:.1f} min")
        if h2.name == hyp.name or (g2 >= g + 2.0 and h2.score >= hyp.score):
            print(f"[{sid}] using the full-resolution result")
            adaptive.append((sid, fx2, h2, r2))
        else:
            print(f"[{sid}] name change not convincing: keeping stride 2")
            adaptive.append((sid, fixes, hyp, ranked))
    print(f"\nadaptive pass: {(time.time() - _t) / 60:.2f} min")
    save_csv(adaptive, DATA, os.path.join(OUT_DIR, "submission_adaptive_hires.csv"))
    if IN_COLAB:
        from google.colab import files
        files.download(os.path.join(OUT_DIR, "submission_adaptive_hires.csv"))

In [ ]:
if IN_NOTEBOOK:
    wide = tuned(params, search_stride=2, search_peaks_each=8, search_keep=1800, screen_keep=180,
                 mem_budget_gb=4.0, search_batch=6, corr_batch=16384)
    print("wide recall:", wide.search_peaks_each, "peaks/template,", wide.search_keep, "candidates,", wide.screen_keep, "screened")
    _t = time.time()
    wide_results = run_validation(DATA, read_figures(DATA, wide), wide, out_path=os.path.join(OUT_DIR, "submission_wide.csv"))
    sub_wide = pd.read_csv(os.path.join(OUT_DIR, "submission_wide.csv"))
    print(sub_wide.shape)
    print(sub_wide["constellation"].value_counts())
    print(f"runtime: {(time.time() - _t) / 60:.2f} min")
    if IN_COLAB:
        from google.colab import files
        files.download(os.path.join(OUT_DIR, "submission_wide.csv"))

## 13. Limitations

- **Bright-tile prior.** Declaring every bright-star tile present earns localisation and geometric credit, but it makes the absent bright tiles false positives.
- **Blown-out tiles.** They cannot be correlated, and they depend on being placed on the named figure. A bright tile whose star is not among the 500 brightest cannot be found at all.
- **Small figures.** Figures with 5 or fewer nodes are hard to name from 350 stars. Random subsets of the stars fit them completely, and the scrambled null scores as high as the truth.
- **Completeness term.** It assumes every in-image figure star is among the 600 brightest detections.
- **Zoom prior.** Tile zooms outside 0.7 to 1.45 would be missed.
- **Train-based settings.** The presence thresholds and the bright-tile settings (refine 60 candidates, shift-search 6) rest on only 3 labelled scenes.

## Course concepts used

| stage | concept |
|---|---|
| pixel histogram and CDF (plot A) | intensity distributions, lookup tables |
| background model | box-filter downsampling, median filter for outliers, piecewise-flat prior |
| border handling | padding |
| `sign(x)·sqrt(|x|)` stretch | point intensity transforms |
| smoothing | separable Gaussian filter |
| halo removal | subtracting the image's own radial profile |
| star detection (plot B) | matched filtering by cross-correlation, LoG/DoG blobs |
| peak picking | non-maximum suppression by dilation |
| centroids | box sums as sliding dot products |
| tile descriptor | detect, describe, match; rotation and scale invariance |
| drawings to point sets | thresholding and normalisation |
| whole-sky search (plot C) | template matching over rotations, `TM_CCOEFF_NORMED` |
| neighbour votes | brute-force keypoint comparison |
| coarse-to-fine search | refinement across position and scale |
| presence rule (plot D) | ratio test, double threshold |
| naming (plot E) | solving `p1 = T·p2` under outliers, RANSAC-style verification |
| placement | search restricted to keypoints |
| beyond the lectures | MAD noise scale, random-pose null, likelihood-ratio score, scrambled-figure null, connected components, distance transform |

In [ ]:
if __name__ == "__main__" and not IN_NOTEBOOK:
    import argparse
    cli = argparse.ArgumentParser(description="Constellation Detection (CS-GY 6643 Project 1, Q5)")
    cli.add_argument("--data", default=None, help="participant/ folder (default: auto-detect)")
    cli.add_argument("--train-eval", action="store_true", help="score the labelled training scenes")
    cli.add_argument("--predict", action="store_true", help="run the validation scenes and write the CSV")
    cli.add_argument("--out", default="submission.csv")
    cli.add_argument("--stride", type=int, default=None, help="search stride (default 1 on GPU, 2 on CPU)")
    args = cli.parse_args()
    if args.data:
        DATA = os.path.abspath(args.data)
    assert DATA and os.path.isdir(os.path.join(DATA, "patterns")), "participant/ folder not found: pass --data"
    params = tuned(search_stride=args.stride or (1 if DEVICE.type == "cuda" else 2), mem_budget_gb=4.0, search_batch=6, corr_batch=16384)
    figures = read_figures(DATA, params)
    if args.train_eval:
        evaluate_train(DATA, figures, params, show_tiles=True, tune=False)
    if args.predict or not args.train_eval:
        run_validation(DATA, figures, params, out_path=args.out)